# Solana Anomaly Detection — Training Notebook

Loads the three monthly parquet files, builds the transaction-level feature matrix,
trains Isolation Forest + Feature-Attention Transformer AE, applies conformal prediction,
and classifies each flagged wallet into an anomaly archetype.

**No BigQuery queries.** All input is read from `/home/ubuntu/data/chunks/`.

## Experiment Suite

| # | Section | Goal |
|---|---------|------|
| 2 | Ablation | Measures each component's contribution vs baseline |
| 3 | Sensitivity | Measures stability of baseline across hyperparameter ranges |
| 4 | Contamination | Measures baseline detection rate on synthetic anomalies |
| 5 | Interpretability | Explains what the baseline flagged and why |

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import db_dtypes                          # registers dbdate parquet extension type
from pandas.api.types import is_datetime64_any_dtype

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from scipy.stats import spearmanr

import matplotlib.pyplot as plt
import seaborn as sns

# Ensure plots render inline in notebook environments (e.g., Kaggle).
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic('matplotlib', 'inline')
except Exception:
    pass

sns.set_theme(style='whitegrid', context='notebook', palette='deep')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
# ── Load parquets ─────────────────────────────────────────────────────────────
CHUNK_DIR = '/kaagle/input/datasets/nschmidtmartinez/anomaly-detect'
names     = ['sep', 'oct', 'nov']

df_flat = pd.concat([
    pd.read_parquet(os.path.join(CHUNK_DIR, f'{m}.parquet'))
    for m in names
], ignore_index=True)

print(f'Loaded : {len(df_flat):,} rows, {df_flat["wallet"].nunique():,} wallets')
print(f'Columns: {list(df_flat.columns)}')

In [ ]:
# ── Timestamp + sort ──────────────────────────────────────────────────────────
if not is_datetime64_any_dtype(df_flat['block_timestamp']):
    df_flat['block_timestamp'] = pd.to_datetime(
        df_flat['block_timestamp'], utc=True, errors='coerce')
elif getattr(df_flat['block_timestamp'].dt, 'tz', None) is None:
    df_flat['block_timestamp'] = df_flat['block_timestamp'].dt.tz_localize('UTC')

df_flat = df_flat.sort_values(['wallet', 'block_timestamp']).reset_index(drop=True)

# Per-tx time since wallet's previous transaction (0 for first tx)
df_flat['delta_time'] = (
    df_flat.groupby('wallet')['block_timestamp']
    .diff().dt.total_seconds().fillna(0.0)
)
print('Timestamps OK. delta_time computed.')

In [ ]:
# ── EDA: Dataset characterisation ────────────────────────────────────────────
print('=== Dataset Overview ===')
print(f'Date range : {df_flat["block_timestamp"].min()} -> {df_flat["block_timestamp"].max()}')
print(f'Rows       : {len(df_flat):,}')
print(f'Wallets    : {df_flat["wallet"].nunique():,}')

# Transactions per wallet
tx_per_wallet = df_flat.groupby('wallet').size()
print('\nTransactions per wallet:')
print(tx_per_wallet.describe(percentiles=[.25, .5, .75, .90, .99]).round(1))

# Monthly breakdown with wallet turnover metrics
df_flat['_month'] = df_flat['block_timestamp'].dt.to_period('M')
monthly_vol = (
    df_flat.groupby('_month')
    .agg(n_tx=('wallet', 'count'), n_wallets=('wallet', 'nunique'))
    .reset_index()
)

first_seen_month = (
    df_flat.groupby('wallet')['block_timestamp']
    .min()
    .dt.to_period('M')
    .value_counts()
    .sort_index()
    .rename_axis('_month')
    .reset_index(name='n_new_wallets')
)

monthly_vol = monthly_vol.merge(first_seen_month, on='_month', how='left').fillna({'n_new_wallets': 0})
monthly_vol['tx_per_wallet'] = monthly_vol['n_tx'] / monthly_vol['n_wallets'].clip(lower=1)
monthly_vol['pct_new_wallets'] = 100.0 * monthly_vol['n_new_wallets'] / monthly_vol['n_wallets'].clip(lower=1)

print('\nMonthly breakdown (expanded):')
print(monthly_vol.to_string(index=False))

# Single-tx vs multi-tx wallets
single_tx = (tx_per_wallet == 1).sum()
print(f'\nSingle-tx wallets: {single_tx:,}  ({single_tx / len(tx_per_wallet) * 100:.1f}%)')
print(f'Multi-tx wallets : {(tx_per_wallet > 1).sum():,}')

# Seaborn plots (compact EDA panel; skips daily volume because it is fixed)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

# A) Wallet activity concentration
sns.histplot(tx_per_wallet, bins=60, log_scale=(True, False), color='#4C78A8', ax=axes[0])
axes[0].set_title('Transactions per Wallet (log-x)')
axes[0].set_xlabel('Transactions per wallet')
axes[0].set_ylabel('Wallet count')

# B) Monthly turnover and wallet freshness
_month_plot = monthly_vol.copy()
_month_plot['_month'] = _month_plot['_month'].astype(str)
sns.barplot(data=_month_plot, x='_month', y='tx_per_wallet', color='#F58518', ax=axes[1])
axes[1].set_title('Tx per Active Wallet by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Transactions per active wallet')

ax1b = axes[1].twinx()
sns.lineplot(data=_month_plot, x='_month', y='pct_new_wallets', marker='o', color='#54A24B', linewidth=1.8, ax=ax1b)
ax1b.set_ylabel('New-wallet share (%)')
ax1b.grid(False)

# C) Key feature distributions
key_feats = ['fee_sol', 'net_sol_flow', 'cpi_ratio', 'drain_sol_ratio']
_dist_parts = []
for feat in key_feats:
    _s = df_flat[feat].astype(float)
    vals = _s.clip(_s.quantile(0.01), _s.quantile(0.99))
    _dist_parts.append(pd.DataFrame({'feature': feat, 'value': vals.values}))

dist_df = pd.concat(_dist_parts, ignore_index=True)
sns.boxenplot(data=dist_df, x='feature', y='value', ax=axes[2], color='#72B7B2')
axes[2].set_title('Key Feature Spread (clipped [1%, 99%])')
axes[2].set_xlabel('Feature')
axes[2].set_ylabel('Value')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: eda_overview.png')


In [ ]:

# ── Feature columns ───────────────────────────────────────────────────────────
# 33 transaction-level features.
# Excluded: wallet_activity_rate (wallet constant), is_defi (hard label,
# already implicit in cpi_ratio / unique_programs / token_program_calls).
tx_feature_cols = [
    'fee_sol', 'compute_units_consumed', 'success_flag',
    'num_accounts', 'num_signers', 'num_writable',
    'num_balance_changes', 'net_sol_flow', 'wallet_sol_delta', 'max_balance_change',
    'token_accounts_closed', 'num_pre_token_balances', 'num_post_token_balances',
    'hour_of_day', 'day_of_week',
    'num_token_transfers', 'unique_mints', 'unique_destinations', 'unique_sources',
    'self_transfers', 'total_token_value', 'max_token_value',
    'num_instructions', 'unique_programs', 'unique_instruction_types',
    'inner_instructions', 'token_program_calls', 'has_compute_budget',
    'fanout_ratio', 'cpi_ratio', 'avg_token_transfer', 'drain_sol_ratio',
    'delta_time',
]

for col in tx_feature_cols:
    df_flat[col] = (pd.to_numeric(df_flat[col], errors='coerce')
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0))

X_tx = df_flat[tx_feature_cols].to_numpy(dtype=np.float32)  # (N_tx, 33)
n_inf = np.isinf(X_tx).sum()
n_nan = np.isnan(X_tx).sum()
print(f'X_tx shape : {X_tx.shape}  |  inf={n_inf}  nan={n_nan}')
assert n_inf == 0 and n_nan == 0, "Unexpected inf/nan in feature matrix!"


In [ ]:
# ── Temporal train / val / test split ────────────────────────────────────────
# Split by calendar month: Sep = train, Oct = val, Nov = test.
# This respects the natural monthly boundary in the 3-month dataset and avoids
# the prior bug where strict first-appearance exclusion left val/test nearly empty.
#
# Transaction masks: purely by block_timestamp month.
# Wallet sets: each wallet is assigned to whichever period holds the majority
# of its transactions, so every wallet appears in exactly one split.
# This gives a large, well-calibrated conformal calibration set (Oct wallets).

df_flat = df_flat.sort_values('block_timestamp').reset_index(drop=True)
X_tx    = df_flat[tx_feature_cols].to_numpy(dtype=np.float32)

_month = df_flat['block_timestamp'].dt.month  # int: 9, 10, 11

train_mask_tx = (_month == 9).values
val_mask_tx   = (_month == 10).values
test_mask_tx  = (_month == 11).values

# Wallet-level split: assign each wallet to its majority-transaction month
_wallet_month_counts = (
    df_flat.groupby(['wallet', _month.rename('month')])
    .size()
    .reset_index(name='n')
)
_wallet_split = (
    _wallet_month_counts
    .sort_values('n', ascending=False)
    .drop_duplicates('wallet')
    .set_index('wallet')['month']
)
train_wallets = set(_wallet_split[_wallet_split == 9].index)
val_wallets   = set(_wallet_split[_wallet_split == 10].index)
test_wallets  = set(_wallet_split[_wallet_split == 11].index)

print(f'Train : {train_mask_tx.sum():,} txs | {len(train_wallets):,} wallets  (Sep)')
print(f'Val   : {val_mask_tx.sum():,} txs | {len(val_wallets):,} wallets  (Oct)')
print(f'Test  : {test_mask_tx.sum():,} txs | {len(test_wallets):,} wallets  (Nov)')
assert len(train_wallets & val_wallets) == 0, "Split overlap!"
assert len(train_wallets & test_wallets) == 0, "Split overlap!"
assert len(val_wallets   & test_wallets) == 0, "Split overlap!" 

In [ ]:

# ── Normalisation (fit on train only) ─────────────────────────────────────────
scaler    = RobustScaler()
X_tx_norm = scaler.fit(X_tx[train_mask_tx]).transform(X_tx).astype(np.float32)

# Clip extreme outliers post-scaling to prevent gradient explosion in the AE
X_tx_norm = np.clip(X_tx_norm, -20.0, 20.0)

n_inf = np.isinf(X_tx_norm).sum()
n_nan = np.isnan(X_tx_norm).sum()
print(f'Normalised  |  inf={n_inf}  nan={n_nan}')
assert n_inf == 0 and n_nan == 0, "inf/nan after normalisation — check raw features!"

print('Train median should be ~0:')
print(pd.DataFrame(X_tx_norm[train_mask_tx], columns=tx_feature_cols)
      .describe().loc[['mean','50%','std']].round(3))


In [ ]:
# ── Feature collinearity & VIF analysis ──────────────────────────────────────
import warnings

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    _HAS_SM = True
except ImportError:
    _HAS_SM = False
    print('statsmodels not found — skipping VIF (pip install statsmodels to enable).')

# Pearson correlation on train split (normalised)
_corr = pd.DataFrame(X_tx_norm[train_mask_tx], columns=tx_feature_cols).corr()

# High-correlation pairs (|r| > 0.80)
_hc = []
for _i in range(len(tx_feature_cols)):
    for _j in range(_i + 1, len(tx_feature_cols)):
        _r = _corr.iloc[_i, _j]
        if abs(_r) > 0.80:
            _hc.append({'feat_a': tx_feature_cols[_i],
                        'feat_b': tx_feature_cols[_j],
                        'pearson_r': round(float(_r), 3)})
hc_df = (pd.DataFrame(_hc)
         .sort_values('pearson_r', ascending=False, key=abs)
         .reset_index(drop=True))
print('Highly correlated pairs (|r| > 0.80):')
print(hc_df.to_string(index=False) if len(hc_df) else '  None found.')

# VIF
if _HAS_SM:
    _rng0 = np.random.default_rng(42)
    _n    = train_mask_tx.sum()
    _samp = _rng0.choice(_n, size=min(8000, _n), replace=False)
    _Xv   = X_tx_norm[train_mask_tx][_samp]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _vif = [variance_inflation_factor(_Xv, _k) for _k in range(_Xv.shape[1])]
    vif_df = (pd.DataFrame({'feature': tx_feature_cols, 'VIF': _vif})
              .sort_values('VIF', ascending=False))
    print('\nVariance Inflation Factors (top 15):')
    print(vif_df.head(15).to_string(index=False))
    print(f'Features with VIF > 10: {vif_df[vif_df.VIF > 10]["feature"].tolist()}')

# Heatmap (Seaborn)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    _corr,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    xticklabels=tx_feature_cols,
    yticklabels=tx_feature_cols,
    cbar_kws={'shrink': 0.75, 'label': 'Pearson r'},
    ax=ax,
)
ax.set_title('Feature Pearson Correlation Matrix (train split)', fontsize=11)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)
plt.tight_layout()
plt.savefig('feature_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: feature_correlation_heatmap.png')


In [ ]:
# ── Isolation Forest ──────────────────────────────────────────────────────────
# Trains on ~546k raw transaction rows.
# max_samples=256 keeps each tree tractable regardless of dataset size.

IF_CFG = dict(n_estimators=300, max_samples=256,
              contamination='auto', random_state=42, n_jobs=-1)

print('Fitting Isolation Forest...')
iforest = IsolationForest(**IF_CFG).fit(X_tx_norm[train_mask_tx])  # fixed: use normalised features (consistent with AE)

if_raw      = -iforest.decision_function(X_tx_norm)           # (N_tx,) higher = more anomalous
if_tx_score = (if_raw - if_raw.min()) / (if_raw.max() - if_raw.min())
print(f'IF done. Score range: [{if_tx_score.min():.4f}, {if_tx_score.max():.4f}]')

In [ ]:

# ── Transformer AE ────────────────────────────────────────────────────────────
# Feature-attention: each of the 33 scalar features = one token.
# Encoder learns cross-feature co-occurrence patterns.
# High reconstruction error = feature combination the model rarely saw in training.

class TxTransformerAE(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=2, ff_dim=128, dropout=0.1):
        super().__init__()
        self.feat_proj = nn.Linear(1, d_model)
        self.pos_embed = nn.Embedding(n_features, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation='gelu')
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, 1)

    def forward(self, x):                          # x: (B, F)
        B, F = x.shape
        pos  = torch.arange(F, device=x.device)
        tok  = self.feat_proj(x.unsqueeze(-1)) + self.pos_embed(pos).unsqueeze(0)
        z    = self.encoder(tok)
        return self.out_proj(z).squeeze(-1)        # (B, F)


TR_CFG = dict(d_model=64, nhead=4, num_layers=2, ff_dim=128, dropout=0.10,
              lr=1e-3, weight_decay=1e-4, epochs=6, batch_size=256)

def train_tx_ae(X_train, X_val, cfg, device='cpu'):
    """Train Transformer AE and return (model, train_losses, val_losses)."""
    model = TxTransformerAE(
        X_train.shape[1], cfg['d_model'], cfg['nhead'],
        cfg['num_layers'], cfg['ff_dim'], cfg['dropout']).to(device)
    opt = torch.optim.Adam(model.parameters(),
                           lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    dl  = DataLoader(TensorDataset(torch.from_numpy(X_train)),
                     batch_size=cfg['batch_size'], shuffle=True)
    train_losses, val_losses = [], []
    for epoch in range(cfg['epochs']):
        model.train()
        total = 0
        for (xb,) in dl:
            xb   = xb.to(device)
            loss = nn.functional.mse_loss(model(xb), xb)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            if torch.isnan(loss):
                raise RuntimeError(
                    f'NaN loss at epoch {epoch+1} — check input data for inf/nan.')
            total += loss.item()
        train_loss = total / len(dl)
        train_losses.append(train_loss)
        # validation loss (no grad, batched)
        model.eval()
        with torch.no_grad():
            vtot, vn = 0.0, 0
            for i in range(0, len(X_val), cfg['batch_size']):
                xb = torch.from_numpy(X_val[i:i+cfg['batch_size']]).to(device)
                vtot += nn.functional.mse_loss(model(xb), xb).item()
                vn   += 1
            val_loss = vtot / max(vn, 1)
        val_losses.append(val_loss)
        print(f'  Epoch {epoch+1}/{cfg["epochs"]}  train={train_loss:.6f}  val={val_loss:.6f}')
    return model, train_losses, val_losses


def score_tx_ae(model, X_all, device='cpu', batch_size=1024):
    """Returns (N_tx,) scalar MSE and (N_tx, F) per-feature squared error."""
    model.eval()
    mse_list, feat_err_list = [], []
    with torch.no_grad():
        for i in range(0, len(X_all), batch_size):
            xb   = torch.from_numpy(X_all[i:i+batch_size]).to(device)
            pred = model(xb)
            fe   = (pred - xb) ** 2
            mse_list.append(fe.mean(dim=1).cpu().numpy())
            feat_err_list.append(fe.cpu().numpy())
    return np.concatenate(mse_list), np.concatenate(feat_err_list)


print('Training Transformer AE...')
tr_model, ae_train_losses, ae_val_losses = train_tx_ae(X_tx_norm[train_mask_tx], X_tx_norm[val_mask_tx], TR_CFG, device)
tr_raw, per_feat_errors = score_tx_ae(tr_model, X_tx_norm, device)  # (N_tx,), (N_tx, 33)
tr_tx_score = (tr_raw - tr_raw.min()) / (tr_raw.max() - tr_raw.min())
print(f'TR done. Score range: [{tr_tx_score.min():.4f}, {tr_tx_score.max():.4f}]')


In [ ]:
# ── AE convergence curves & hyperparameter selection ─────────────────────────
# Compare three architecture configs on val MSE; confirm "default" is reasonable.

_AE_GRID = [
    dict(label='small',   d_model=32,  nhead=4, num_layers=1, ff_dim=64,  dropout=0.10),
    dict(label='default', d_model=64,  nhead=4, num_layers=2, ff_dim=128, dropout=0.10),
    dict(label='large',   d_model=128, nhead=4, num_layers=3, ff_dim=256, dropout=0.15),
]
_BASE = dict(lr=TR_CFG['lr'], weight_decay=TR_CFG['weight_decay'],
             epochs=TR_CFG['epochs'], batch_size=TR_CFG['batch_size'])

_ae_sel_rows = []
_best_val    = np.inf
_best_label  = 'default'

for _arch in _AE_GRID:
    _lbl = _arch['label']
    _cfg = {k: v for k, v in _arch.items() if k != 'label'}
    _cfg.update(_BASE)
    print(f'  Fitting AE [{_lbl}] ...')
    _m, _tl, _vl = train_tx_ae(X_tx_norm[train_mask_tx],
                                X_tx_norm[val_mask_tx], _cfg, device)
    _ae_sel_rows.append({'config': _lbl,
                         'val_mse': round(float(_vl[-1]), 6),
                         'train_mse': round(float(_tl[-1]), 6),
                         'd_model': _arch['d_model'],
                         'num_layers': _arch['num_layers']})
    if _vl[-1] < _best_val:
        _best_val   = _vl[-1]
        _best_label = _lbl

ae_sel_df = pd.DataFrame(_ae_sel_rows)
print('\nAE configuration comparison:')
print(ae_sel_df.to_string(index=False))
print(f'\nSelected config: "{_best_label}"  (val_mse={_best_val:.6f})')
if _best_label != 'default':
    print(f'  NOTE: re-run cell 8 with d_model/num_layers from "{_best_label}" for best results.')

# Convergence plot for the "default" config (already trained as tr_model)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

_conv_df = pd.DataFrame({
    'epoch': np.arange(1, len(ae_train_losses) + 1),
    'train_mse': ae_train_losses,
    'val_mse': ae_val_losses,
}).melt(id_vars='epoch', var_name='split', value_name='mse')

sns.lineplot(data=_conv_df, x='epoch', y='mse', hue='split', marker='o', ax=axes[0])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_title('AE Convergence (default config)')

sns.barplot(data=ae_sel_df, x='config', y='val_mse', ax=axes[1])
axes[1].set_xlabel('Config')
axes[1].set_ylabel('Val MSE')
axes[1].set_title('AE Config Comparison (Val MSE)')

plt.tight_layout()
plt.savefig('ae_convergence_and_selection.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: ae_convergence_and_selection.png')


### AE model selection (Chollet-style rigour)

Per the **universal workflow of ML** (Chollet, *Deep Learning with Python*): after beating a baseline, we should *regularise and tune* by searching hyperparameters rather than comparing only a few hand-picked configs. The section below first runs a compact **capacity + regularised diagnostics** pass (small/medium/large plus regularised variants), then a full **Optuna optimisation** study over AE and IF settings.

In [ ]:
# ── Capacity + regularised diagnostics (pre-Optuna) ──────────────────────────
# Chollet-style sanity check: compare model capacity and regularisation before
# full hyperparameter optimisation.

if 'train_tx_ae' not in globals() or 'score_tx_ae' not in globals() or 'TR_CFG' not in globals():
    raise RuntimeError('Run AE setup first (`train_tx_ae`, `score_tx_ae`, `TR_CFG` required).')

if 'X_tx_norm' not in globals() or 'train_mask_tx' not in globals() or 'val_mask_tx' not in globals():
    raise RuntimeError('Missing data splits (`X_tx_norm`, `train_mask_tx`, `val_mask_tx`).')

DIAG_EPOCHS = int(min(TR_CFG.get('epochs', 10), 6))
DIAG_BATCH_SIZE = int(TR_CFG.get('batch_size', 256))
ALPHA_DIAG = float(ALPHA) if 'ALPHA' in globals() else 0.01

_base_lr = float(TR_CFG.get('lr', 1e-3))
_base_wd = float(TR_CFG.get('weight_decay', 1e-5))
_base_dropout = float(TR_CFG.get('dropout', 0.10))

_diag_cfgs = [
    dict(profile='small', d_model=32, nhead=4, num_layers=1, ff_dim=64,
         dropout=0.10, lr=_base_lr, weight_decay=_base_wd),
    dict(profile='medium', d_model=64, nhead=4, num_layers=2, ff_dim=128,
         dropout=max(0.10, _base_dropout), lr=_base_lr, weight_decay=_base_wd),
    dict(profile='large', d_model=128, nhead=8, num_layers=3, ff_dim=256,
         dropout=0.15, lr=_base_lr, weight_decay=_base_wd),
    dict(profile='medium_regularised', d_model=64, nhead=4, num_layers=2, ff_dim=128,
         dropout=max(0.20, _base_dropout), lr=_base_lr, weight_decay=max(_base_wd * 5.0, 1e-4)),
    dict(profile='large_regularised', d_model=128, nhead=8, num_layers=3, ff_dim=256,
         dropout=0.25, lr=_base_lr * 0.7, weight_decay=max(_base_wd * 8.0, 2e-4)),
]

_diag_rows = []
for _cfg_base in _diag_cfgs:
    _cfg = dict(_cfg_base)
    _cfg['epochs'] = DIAG_EPOCHS
    _cfg['batch_size'] = DIAG_BATCH_SIZE

    print(
        f"Diagnostic run [{_cfg['profile']}]: d={_cfg['d_model']} h={_cfg['nhead']} "
        f"L={_cfg['num_layers']} ff={_cfg['ff_dim']} dr={_cfg['dropout']:.2f} "
        f"lr={_cfg['lr']:.2e} wd={_cfg['weight_decay']:.2e}"
    )

    _m, _train_losses, _val_losses = train_tx_ae(
        X_tx_norm[train_mask_tx],
        X_tx_norm[val_mask_tx],
        _cfg,
        device,
    )

    _tr_raw_diag, _ = score_tx_ae(_m, X_tx_norm, device)
    _tr_score_diag = (_tr_raw_diag - _tr_raw_diag.min()) / (_tr_raw_diag.max() - _tr_raw_diag.min() + 1e-12)
    _val_sc = _tr_score_diag[val_mask_tx]

    _tail = float(np.quantile(_val_sc, 0.99) - np.quantile(_val_sc, 0.50))
    _thr = float(np.quantile(_val_sc, 1 - ALPHA_DIAG))

    _diag_rows.append({
        'profile': _cfg['profile'],
        'regularised': ('regularised' in _cfg['profile']),
        'train_mse': float(_train_losses[-1]),
        'val_mse': float(_val_losses[-1]),
        'generalisation_gap': float(_val_losses[-1] - _train_losses[-1]),
        'val_tail_contrast': _tail,
        'val_threshold': _thr,
        'd_model': int(_cfg['d_model']),
        'nhead': int(_cfg['nhead']),
        'num_layers': int(_cfg['num_layers']),
        'dropout': float(_cfg['dropout']),
        'lr': float(_cfg['lr']),
        'weight_decay': float(_cfg['weight_decay']),
        'epochs': int(_cfg['epochs']),
        'batch_size': int(_cfg['batch_size']),
    })

chollet_capacity_regularised_df = (
    pd.DataFrame(_diag_rows)
    .sort_values(['val_mse', 'generalisation_gap', 'val_tail_contrast'], ascending=[True, True, False])
    .reset_index(drop=True)
)

print('\nCapacity + regularised diagnostics:')
print(chollet_capacity_regularised_df.to_string(index=False))

_best_profile = chollet_capacity_regularised_df.iloc[0]['profile']
print(f"\nRecommended centre for Optuna optimisation: {_best_profile}")
print('Saved table: `chollet_capacity_regularised_df`.')

In [ ]:
# ── Optuna joint hyperparameter search (AE + IF) ─────────────────────────────
# Replaces random search with Bayesian optimisation (TPE).
# Tunes exactly the gap parameters:
#   AE architecture: nhead, num_layers, dropout (+ d_model/ff_dim for validity)
#   AE training: lr, weight_decay, epochs
#   IF: n_estimators, max_samples

try:
    import optuna
except Exception as _e:
    raise ImportError(
        'Optuna is not installed. Run `pip install optuna` and re-run this cell.'
    ) from _e

from sklearn.ensemble import IsolationForest

_REQUIRED = [
    'train_tx_ae', 'score_tx_ae', 'TR_CFG', 'IF_CFG',
    'X_tx', 'X_tx_norm', 'train_mask_tx', 'val_mask_tx', 'scaler'
]
_missing = [v for v in _REQUIRED if v not in globals()]
if _missing:
    raise RuntimeError(f'Missing prerequisites for Optuna search: {_missing}')

optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_N_TRIALS = 20
OPTUNA_TIMEOUT_SEC = None
OPTUNA_APPLY_BEST = True  # Set True to overwrite IF/TR models + scores.

_ALPHA_EVAL = float(ALPHA) if 'ALPHA' in globals() else 0.01
_W_IF_EVAL = float(W_IF) if 'W_IF' in globals() else 0.5
_W_TR_EVAL = float(W_TR) if 'W_TR' in globals() else 0.5

_w_sum = _W_IF_EVAL + _W_TR_EVAL
if _w_sum <= 0:
    _W_IF_EVAL, _W_TR_EVAL = 0.5, 0.5
else:
    _W_IF_EVAL, _W_TR_EVAL = _W_IF_EVAL / _w_sum, _W_TR_EVAL / _w_sum

_if_train_n = int(np.sum(train_mask_tx))
if _if_train_n == 0 or int(np.sum(val_mask_tx)) == 0:
    raise RuntimeError('Need non-empty train/val transaction splits for Optuna search.')

# Fixed synthetic anomaly batch for deterministic trial comparison.
_syn_rng = np.random.default_rng(42)
_feat_idx = {f: i for i, f in enumerate(tx_feature_cols)}
_val_pool = np.where(val_mask_tx)[0]
_n_syn = int(min(3000, len(_val_pool)))
if _n_syn == 0:
    raise RuntimeError('No validation transactions available for synthetic objective.')

_syn_pick = _syn_rng.choice(_val_pool, size=_n_syn, replace=False)
X_syn_raw_optuna = X_tx[_syn_pick].copy()

for _f in ['fee_sol', 'compute_units_consumed', 'fanout_ratio', 'cpi_ratio', 'drain_sol_ratio', 'inner_instructions']:
    if _f in _feat_idx:
        _j = _feat_idx[_f]
        X_syn_raw_optuna[:, _j] = X_syn_raw_optuna[:, _j] * _syn_rng.uniform(2.5, 6.0, size=_n_syn)

if 'success_flag' in _feat_idx:
    X_syn_raw_optuna[:, _feat_idx['success_flag']] = 0.0

_train_std = X_tx[train_mask_tx].std(axis=0) + 1e-6
X_syn_raw_optuna += _syn_rng.normal(0.0, 1.25 * _train_std, size=X_syn_raw_optuna.shape)
X_syn_norm_optuna = np.clip(scaler.transform(X_syn_raw_optuna).astype(np.float32), -20.0, 20.0)

_base_if_cfg = dict(IF_CFG)
_base_if_cfg.setdefault('contamination', 0.005)
_base_if_cfg.setdefault('random_state', 42)
_base_if_cfg.setdefault('n_jobs', -1)


def _optuna_objective(trial):
    d_model = int(trial.suggest_categorical('ae_d_model', [32, 64, 96, 128]))
    nhead_choices = [h for h in [2, 4, 8] if d_model % h == 0]
    nhead = int(trial.suggest_categorical('ae_nhead', nhead_choices))

    num_layers = int(trial.suggest_int('ae_num_layers', 1, 4))
    dropout = float(trial.suggest_float('ae_dropout', 0.05, 0.30, step=0.05))

    lr = float(trial.suggest_float('ae_lr', 1e-4, 3e-3, log=True))
    weight_decay = float(trial.suggest_float('ae_weight_decay', 1e-7, 1e-3, log=True))
    epochs = int(trial.suggest_int('ae_epochs', 4, 16))

    ff_mult = int(trial.suggest_int('ae_ff_mult', 2, 4))
    ff_dim = int(d_model * ff_mult)

    if_n_estimators = int(trial.suggest_int('if_n_estimators', 150, 600, step=50))
    if_max_samples_raw = int(trial.suggest_categorical('if_max_samples', [128, 256, 512, 1024]))
    if_max_samples = int(min(if_max_samples_raw, _if_train_n))

    tr_cfg = dict(TR_CFG)
    tr_cfg.update({
        'd_model': d_model,
        'nhead': nhead,
        'num_layers': num_layers,
        'ff_dim': ff_dim,
        'dropout': dropout,
        'lr': lr,
        'weight_decay': weight_decay,
        'epochs': epochs,
    })

    if_cfg = dict(_base_if_cfg)
    if_cfg.update({
        'n_estimators': if_n_estimators,
        'max_samples': if_max_samples,
    })

    if_model = IsolationForest(**if_cfg)
    if_model.fit(X_tx_norm[train_mask_tx])

    if_raw_trial = -if_model.decision_function(X_tx_norm)
    if_score_trial = (if_raw_trial - if_raw_trial.min()) / (if_raw_trial.max() - if_raw_trial.min() + 1e-12)

    tr_model_trial, tr_train_losses, tr_val_losses = train_tx_ae(
        X_tx_norm[train_mask_tx],
        X_tx_norm[val_mask_tx],
        tr_cfg,
        device,
    )
    tr_raw_trial, _ = score_tx_ae(tr_model_trial, X_tx_norm, device)
    tr_score_trial = (tr_raw_trial - tr_raw_trial.min()) / (tr_raw_trial.max() - tr_raw_trial.min() + 1e-12)

    comb_tx_trial = _W_IF_EVAL * if_score_trial + _W_TR_EVAL * tr_score_trial
    val_scores = comb_tx_trial[val_mask_tx]

    threshold = float(np.quantile(val_scores, 1 - _ALPHA_EVAL))
    tail_contrast = float(np.quantile(val_scores, 0.99) - np.quantile(val_scores, 0.50))

    if_syn_raw = -if_model.decision_function(X_syn_norm_optuna)
    if_syn = (if_syn_raw - if_raw_trial.min()) / (if_raw_trial.max() - if_raw_trial.min() + 1e-12)

    tr_syn_raw, _ = score_tx_ae(tr_model_trial, X_syn_norm_optuna, device)
    tr_syn = (tr_syn_raw - tr_raw_trial.min()) / (tr_raw_trial.max() - tr_raw_trial.min() + 1e-12)

    comb_syn = _W_IF_EVAL * if_syn + _W_TR_EVAL * tr_syn
    syn_detect_rate = float(np.mean(comb_syn >= threshold))

    val_mse = float(tr_val_losses[-1]) if len(tr_val_losses) else np.nan

    # Composite no-label objective (maximise):
    #   + synthetic detectability (dominant)
    #   + anomaly-tail contrast on validation
    #   - reconstruction loss penalty
    objective = float(
        0.70 * syn_detect_rate
        + 0.20 * tail_contrast
        - 0.10 * np.log1p(max(val_mse, 1e-12))
    )

    trial.set_user_attr('syn_detect_rate', syn_detect_rate)
    trial.set_user_attr('tail_contrast', tail_contrast)
    trial.set_user_attr('val_mse', val_mse)
    trial.set_user_attr('threshold', threshold)
    trial.set_user_attr('if_max_samples_effective', if_max_samples)

    return objective


_sampler = optuna.samplers.TPESampler(seed=42, multivariate=True)
optuna_study = optuna.create_study(direction='maximize', sampler=_sampler, study_name='ae_if_joint_search')

print(f'Running Optuna study: n_trials={OPTUNA_N_TRIALS}, timeout={OPTUNA_TIMEOUT_SEC}')
optuna_study.optimize(
    _optuna_objective,
    n_trials=OPTUNA_N_TRIALS,
    timeout=OPTUNA_TIMEOUT_SEC,
    gc_after_trial=True,
    catch=(RuntimeError, ValueError),
)

_opt_rows = []
for _t in optuna_study.trials:
    if _t.value is None:
        continue
    _opt_rows.append({
        'trial': int(_t.number),
        'objective': float(_t.value),
        'syn_detect_rate': float(_t.user_attrs.get('syn_detect_rate', np.nan)),
        'tail_contrast': float(_t.user_attrs.get('tail_contrast', np.nan)),
        'val_mse': float(_t.user_attrs.get('val_mse', np.nan)),
        **_t.params,
    })

if len(_opt_rows) == 0:
    raise RuntimeError('All Optuna trials failed. Lower search ranges or check GPU/CPU memory.')

optuna_joint_search_df = pd.DataFrame(_opt_rows).sort_values('objective', ascending=False).reset_index(drop=True)
print('\nTop Optuna trials (joint AE + IF):')
print(optuna_joint_search_df.head(10).to_string(index=False))

_best = optuna_study.best_trial
optuna_best_params = dict(_best.params)

optuna_best_tr_cfg = dict(TR_CFG)
optuna_best_tr_cfg.update({
    'd_model': int(optuna_best_params['ae_d_model']),
    'nhead': int(optuna_best_params['ae_nhead']),
    'num_layers': int(optuna_best_params['ae_num_layers']),
    'ff_dim': int(int(optuna_best_params['ae_d_model']) * int(optuna_best_params['ae_ff_mult'])),
    'dropout': float(optuna_best_params['ae_dropout']),
    'lr': float(optuna_best_params['ae_lr']),
    'weight_decay': float(optuna_best_params['ae_weight_decay']),
    'epochs': int(optuna_best_params['ae_epochs']),
})

optuna_best_if_cfg = dict(_base_if_cfg)
optuna_best_if_cfg.update({
    'n_estimators': int(optuna_best_params['if_n_estimators']),
    'max_samples': int(min(int(optuna_best_params['if_max_samples']), _if_train_n)),
})

print('\nBest trial objective:', float(_best.value))
print('Best IF config:', optuna_best_if_cfg)
print('Best AE config:', optuna_best_tr_cfg)

if OPTUNA_APPLY_BEST:
    print('\nApplying best Optuna hyperparameters and recomputing scores...')

    IF_CFG = dict(optuna_best_if_cfg)
    iforest = IsolationForest(**IF_CFG).fit(X_tx_norm[train_mask_tx])
    if_raw = -iforest.decision_function(X_tx_norm)
    if_tx_score = (if_raw - if_raw.min()) / (if_raw.max() - if_raw.min() + 1e-12)

    TR_CFG = dict(optuna_best_tr_cfg)
    tr_model, ae_train_losses, ae_val_losses = train_tx_ae(
        X_tx_norm[train_mask_tx],
        X_tx_norm[val_mask_tx],
        TR_CFG,
        device,
    )
    tr_raw, per_feat_errors = score_tx_ae(tr_model, X_tx_norm, device)
    tr_tx_score = (tr_raw - tr_raw.min()) / (tr_raw.max() - tr_raw.min() + 1e-12)

    print('Updated `IF_CFG`, `TR_CFG`, `if_tx_score`, `tr_tx_score`, and model objects.')
else:
    print('\nOPTUNA_APPLY_BEST=False: results saved as `optuna_joint_search_df`,\n'
          '`optuna_best_if_cfg`, and `optuna_best_tr_cfg`.')

In [ ]:
# ── Combine scores + aggregate to wallet level ─────────────────────────────────
# max aggregation: one anomalous transaction surfaces the wallet.

W_IF, W_TR = 0.5, 0.5
combined_tx = W_IF * if_tx_score + W_TR * tr_tx_score

df_flat['if_tx_score']       = if_tx_score
df_flat['tr_tx_score']       = tr_tx_score
df_flat['combined_tx_score'] = combined_tx

wallet_scores = df_flat.groupby('wallet').agg(
    if_score      =('if_tx_score',       'max'),
    tr_score      =('tr_tx_score',       'max'),
    raw_score     =('combined_tx_score', 'max'),
    p90_score     =('combined_tx_score', lambda x: np.percentile(x, 90)),
    n_tx          =('combined_tx_score', 'count'),
).reset_index()

print(f'Wallet scores: {len(wallet_scores):,}')
print(wallet_scores[['if_score','tr_score','raw_score']]
      .describe(percentiles=[.5,.9,.99]).round(4))

In [ ]:
# ── Conformal prediction (wallet level) ───────────────────────────────────────
# Calibrate on val wallets; compute p-values for test wallets.
# Theoretical guarantee: FPR ≤ alpha under exchangeability assumption.

ALPHA = 0.01

val_mask_w  = wallet_scores['wallet'].isin(val_wallets)
test_mask_w = wallet_scores['wallet'].isin(test_wallets)

cal_scores  = wallet_scores.loc[val_mask_w,  'raw_score'].values
test_scores = wallet_scores.loc[test_mask_w, 'raw_score'].values

conformal_threshold = np.quantile(cal_scores, 1 - ALPHA)
test_pvals = np.array([
    (np.sum(cal_scores >= s) + 1) / (len(cal_scores) + 1)
    for s in test_scores
])

n_flagged = (test_pvals <= ALPHA).sum()
print(f'Conformal threshold (alpha={ALPHA}): {conformal_threshold:.4f}')
print(f'Test wallets flagged : {n_flagged} / {len(test_scores)} '
      f'({n_flagged/len(test_scores)*100:.2f}%)')

wallet_scores.loc[test_mask_w, 'conformal_p']    = test_pvals
wallet_scores.loc[test_mask_w, 'conformal_flag'] = (test_pvals <= ALPHA).astype(int)

In [ ]:
# ── Conformal validity: empirical coverage & temporal exchangeability ─────────
# Under valid exchangeability, LOO p-values on cal set should be uniform → flag
# rate ≈ alpha. A KS test on score distributions over time checks temporal drift.

from scipy.stats import ks_2samp

# 1) LOO conformal p-values on the calibration (val) set
print(f'Computing LOO p-values on {len(cal_scores):,} cal wallets...')
_cal_loo_p = np.array([
    (np.sum(np.delete(cal_scores, _i) >= cal_scores[_i]) + 1) / len(cal_scores)
    for _i in range(len(cal_scores))
])
_realized_fpr = float((_cal_loo_p <= ALPHA).mean())
print(f'Target alpha              : {ALPHA}')
print(f'Realised flag rate on cal : {_realized_fpr:.4f}  '
      f'({("OK" if abs(_realized_fpr - ALPHA) < 0.01 else "WARN — deviates from alpha")})')

# 2) Temporal exchangeability: KS test across three equal time-thirds of val set
_val_df     = df_flat[df_flat['wallet'].isin(val_wallets)].sort_values('block_timestamp')
_thirds_idx = np.array_split(_val_df.index, 3)
_thirds_sc  = [_val_df.loc[_idx, 'combined_tx_score'].values for _idx in _thirds_idx]
_ks_stat, _ks_p = ks_2samp(_thirds_sc[0], _thirds_sc[2])
print(f'\nKS test (val early vs late third): stat={_ks_stat:.4f}  p={_ks_p:.4e}')
if _ks_p < 0.05:
    print('  WARN: score distribution drifts over time — exchangeability is violated.')
    print('  Recommendation: use weighted conformal prediction (Tibshirani et al., 2019).')
else:
    print('  OK: no significant temporal drift detected.')

# Seaborn plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# p-value histogram (should be uniform)
sns.histplot(_cal_loo_p, bins=20, color='#4C78A8', edgecolor='white', ax=axes[0])
axes[0].axhline(len(_cal_loo_p) / 20, color='red', linestyle='--', label='Expected (uniform)')
axes[0].set_xlabel('LOO Conformal p-value')
axes[0].set_ylabel('Count')
axes[0].set_title('Cal-Set p-value Distribution\n(uniform = valid coverage)')
axes[0].legend()

# CDF by time-third
_cdf_parts = []
for _sc, _lbl in zip(_thirds_sc, ['early', 'mid', 'late']):
    _ss = np.sort(_sc)
    if len(_ss) == 0:
        continue
    _cdf_parts.append(pd.DataFrame({'score': _ss, 'cdf': np.linspace(0, 1, len(_ss)), 'period': _lbl}))

if len(_cdf_parts):
    _cdf_df = pd.concat(_cdf_parts, ignore_index=True)
    sns.lineplot(data=_cdf_df, x='score', y='cdf', hue='period', ax=axes[1])
axes[1].axvline(conformal_threshold, color='black', linestyle='--', linewidth=1, label=f'threshold (a={ALPHA})')
axes[1].set_xlabel('Combined score')
axes[1].set_ylabel('CDF')
axes[1].set_title('Score CDF by Time Period (val)\nDivergence -> exchangeability violation')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('conformal_validity.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: conformal_validity.png')


In [ ]:
# ── Wallet score aggregation strategy comparison ──────────────────────────────
# max, mean, and p90 aggregation may produce meaningfully different flag sets.
# High Jaccard / rank-corr → choice is stable; low → motivates principled selection.

def _conformal_flags_simple(cal_s, test_s, alpha):
    thr = np.quantile(cal_s, 1 - alpha)
    return (test_s >= thr).astype(int)

_AGG_FUNS = {
    'max':  lambda x: x.max(),
    'mean': lambda x: x.mean(),
    'p90':  lambda x: np.percentile(x, 90),
}

_agg_flags  = {}
_agg_scores = {}

for _agg_name, _fn in _AGG_FUNS.items():
    _ws    = df_flat.groupby('wallet')['combined_tx_score'].agg(_fn)
    _cal_s = _ws[_ws.index.isin(val_wallets)].values
    _tst_s = _ws[_ws.index.isin(test_wallets)].values
    _agg_flags[_agg_name]  = _conformal_flags_simple(_cal_s, _tst_s, ALPHA)
    _agg_scores[_agg_name] = _tst_s

_base_fl = _agg_flags['max']
_base_sc = _agg_scores['max']

agg_comp_rows = []
for _name, _fl in _agg_flags.items():
    _inter = int(np.sum((_fl == 1) & (_base_fl == 1)))
    _union = int(np.sum((_fl == 1) | (_base_fl == 1)))
    _jac   = _inter / _union if _union > 0 else 1.0
    _rho   = float(spearmanr(_agg_scores[_name], _base_sc).correlation)
    agg_comp_rows.append({'aggregation': _name,
                          'flagged': int(_fl.sum()),
                          'flag_rate_pct': round(float(_fl.mean() * 100), 2),
                          'jaccard_vs_max': round(_jac, 4),
                          'rank_corr_vs_max': round(_rho, 4)})

wallet_agg_df = pd.DataFrame(agg_comp_rows)
print('Wallet aggregation strategy comparison:')
print(wallet_agg_df.to_string(index=False))
print('\nInterpretation: Jaccard < 0.8 or rank_corr < 0.9 means aggregation choice '
      'materially changes which wallets are flagged.')


In [ ]:
# ── 10. Data-Driven Anomaly Typology (DBSCAN / KMeans) ───────────────────────
# Instead of heuristics, we extract the Transformer's latent embeddings and
# cluster them to organically discover anomaly archetypes.

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from hdbscan.validity import validity_index as dbcv_validity_index
    HAS_DBCV = True
except Exception:
    HAS_DBCV = False

def extract_tx_latent_mean(model, X_all, device='cpu', batch_size=2048):
    """Mean-pool encoder token states -> one embedding per transaction."""
    model.eval()
    out = []
    import torch
    with torch.no_grad():
        for i in range(0, len(X_all), batch_size):
            xb = torch.from_numpy(X_all[i:i + batch_size]).to(device)
            bsz, n_feat = xb.shape
            pos = torch.arange(n_feat, device=xb.device)
            tok = model.feat_proj(xb.unsqueeze(-1)) + model.pos_embed(pos).unsqueeze(0)
            z = model.encoder(tok)                      # (B, F, d_model)
            emb = z.mean(dim=1).cpu().numpy()          # (B, d_model)
            out.append(emb)
    return np.concatenate(out, axis=0)

# One representative transaction per wallet (highest combined score if available)
if 'combined_tx_score' in df_flat.columns:
    rep_tx_idx = df_flat.groupby('wallet')['combined_tx_score'].idxmax()
else:
    rep_rows = df_flat.groupby('wallet', sort=False).head(1)
    rep_tx_idx = pd.Series(rep_rows.index.values, index=rep_rows['wallet'].values)

wallet_rep = rep_tx_idx.index.to_numpy()
idx_rep = rep_tx_idx.to_numpy(dtype=int)

X_rep = X_tx_norm[idx_rep].astype(np.float32)
emb_wallet = extract_tx_latent_mean(tr_model, X_rep, device=device, batch_size=2048)

scaler_emb = StandardScaler()
X_emb = scaler_emb.fit_transform(emb_wallet)

MAX_EVAL_N = 5000
rng = np.random.default_rng(42)
if len(X_emb) > MAX_EVAL_N:
    eval_idx = rng.choice(len(X_emb), size=MAX_EVAL_N, replace=False)
    X_eval = X_emb[eval_idx]
else:
    eval_idx = np.arange(len(X_emb))
    X_eval = X_emb

k_grid = [2, 3, 4, 5, 6, 8, 10, 12]
k_grid = [k for k in k_grid if 2 <= k < len(X_eval)]

rows = []
for k in k_grid:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_eval)
    if len(np.unique(labels)) < 2: continue
    rows.append({
        'k': int(k),
        'silhouette': float(silhouette_score(X_eval, labels)),
        'davies_bouldin': float(davies_bouldin_score(X_eval, labels)),
        'calinski_harabasz': float(calinski_harabasz_score(X_eval, labels)),
    })

if len(rows) > 0:
    transformer_cluster_metrics = pd.DataFrame(rows).sort_values('k').reset_index(drop=True)
    tcm = transformer_cluster_metrics.copy()
    tcm['silhouette_z'] = (tcm['silhouette'] - tcm['silhouette'].mean()) / (tcm['silhouette'].std() + 1e-12)
    tcm['db_inv_z'] = ((-tcm['davies_bouldin']) - (-tcm['davies_bouldin']).mean()) / ((-tcm['davies_bouldin']).std() + 1e-12)
    tcm['ch_z'] = (tcm['calinski_harabasz'] - tcm['calinski_harabasz'].mean()) / (tcm['calinski_harabasz'].std() + 1e-12)
    tcm['quality_score'] = tcm[['silhouette_z', 'db_inv_z', 'ch_z']].mean(axis=1)
    best_k = int(tcm.loc[tcm['quality_score'].idxmax(), 'k'])
    
    print('Transformer embedding clustering metrics:')
    print(tcm[['k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'quality_score']].to_string(index=False))
    print(f"\nRecommended k (composite score): {best_k}")
else:
    best_k = 4
    tcm = pd.DataFrame()  # empty fallback

MAX_DENSITY_EVAL_N = 3000
X_density = X_eval if len(X_eval) <= MAX_DENSITY_EVAL_N else X_eval[rng.choice(len(X_eval), size=MAX_DENSITY_EVAL_N, replace=False)]

knn = NearestNeighbors(n_neighbors=11).fit(X_density)
knn_dist, _ = knn.kneighbors(X_density)
eps_candidates = np.unique(np.quantile(np.sort(knn_dist[:, -1]), [0.70, 0.80, 0.90, 0.95]))

dbscan_rows = []
for min_samples in [5, 10, 20]:
    for eps in eps_candidates:
        db = DBSCAN(eps=float(eps), min_samples=int(min_samples), metric='euclidean', n_jobs=-1)
        labels_db = db.fit_predict(X_density)
        n_clusters = int(len(set(labels_db)) - (1 if -1 in labels_db else 0))
        sil_db = float(silhouette_score(X_density[labels_db != -1], labels_db[labels_db != -1])) if n_clusters >= 2 and np.sum(labels_db != -1) > 10 and len(np.unique(labels_db[labels_db != -1])) >= 2 else np.nan
        dbcv = float(dbcv_validity_index(X_density.astype(np.float64), labels_db)) if HAS_DBCV and n_clusters >= 2 else np.nan
        dbscan_rows.append({'eps': float(eps), 'min_samples': int(min_samples), 'n_clusters': n_clusters, 'noise_frac': float(np.mean(labels_db == -1)), 'silhouette_non_noise': sil_db, 'dbcv': dbcv})

if len(dbscan_rows) > 0:
    dbscan_dbcv_metrics = pd.DataFrame(dbscan_rows).sort_values(['dbcv', 'silhouette_non_noise'], ascending=False).reset_index(drop=True)
    best_dbcv_row = dbscan_dbcv_metrics.iloc[0] if len(dbscan_dbcv_metrics) > 0 and dbscan_dbcv_metrics['n_clusters'].iloc[0] > 1 else None
    
    if HAS_DBCV and dbscan_dbcv_metrics['dbcv'].notna().any() and best_dbcv_row is not None:
        print('\nDBSCAN + DBCV (top configs):')
        print(dbscan_dbcv_metrics.head(10).to_string(index=False))
    else:
        print('\nDBCV unavailable or not finite. Showing DBSCAN diagnostic table (silhouette/non-noise + noise fraction).')
        print(dbscan_dbcv_metrics.head(10).to_string(index=False))
else:
    best_dbcv_row = None
    dbscan_dbcv_metrics = pd.DataFrame()

if best_dbcv_row is not None and best_dbcv_row['n_clusters'] > 1 and best_dbcv_row['noise_frac'] < 0.8:
    print(f"\nUsing DBSCAN: eps={best_dbcv_row['eps']:.4f}, min_samples={int(best_dbcv_row['min_samples'])}")
    best_model = DBSCAN(eps=best_dbcv_row['eps'], min_samples=int(best_dbcv_row['min_samples']), metric='euclidean', n_jobs=-1)
else:
    print(f"\nFalling back to KMeans (k={best_k})")
    best_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)

labels_full = best_model.fit_predict(X_emb)
wallet_embedding_clusters = pd.DataFrame({'wallet': wallet_rep, 'cluster': labels_full})
cluster_map = {c: f'Cluster {c}' if c != -1 else 'Noise' for c in set(labels_full)}
wallet_embedding_clusters['semantic_anomaly_type'] = wallet_embedding_clusters['cluster'].map(cluster_map)

if 'semantic_anomaly_type' in wallet_scores.columns:
    wallet_scores = wallet_scores.drop(columns=['semantic_anomaly_type'])
wallet_scores = wallet_scores.merge(wallet_embedding_clusters[['wallet', 'semantic_anomaly_type']], on='wallet', how='left')
wallet_scores['semantic_anomaly_type'] = wallet_scores['semantic_anomaly_type'].fillna('Unassigned')

if 'semantic_anomaly_type' in df_flat.columns:
    df_flat = df_flat.drop(columns=['semantic_anomaly_type'])
df_flat = df_flat.merge(wallet_embedding_clusters[['wallet', 'semantic_anomaly_type']], on='wallet', how='left')
df_flat['anomaly_type'] = df_flat['semantic_anomaly_type'].fillna('Unassigned')
tx_anomaly_type = df_flat['anomaly_type'].values

print('\nFlagged wallet primary type breakdown (Data-Driven Clusters):')
flagged = wallet_scores[wallet_scores.get('conformal_flag', pd.Series(0)) == 1]
if len(flagged):
    print(flagged['semantic_anomaly_type'].value_counts())

# ── Seaborn plot metric curves (centroid + density perspectives) ──
fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.0))
ax_a, ax_b = axes[0]
ax_c, ax_d = axes[1]

if not tcm.empty:
    sns.lineplot(data=tcm, x='k', y='silhouette', marker='o', ax=ax_a)
    ax_a.axvline(best_k, linestyle='--', linewidth=1, color='black')
    ax_a.set_title('Silhouette (higher better)')
    ax_a.set_xlabel('k')
    ax_a.set_ylabel('Score')

    sns.lineplot(data=tcm, x='k', y='davies_bouldin', marker='o', ax=ax_b)
    ax_b.axvline(best_k, linestyle='--', linewidth=1, color='black')
    ax_b.set_title('Davies-Bouldin (lower better)')
    ax_b.set_xlabel('k')
    ax_b.set_ylabel('Score')

    sns.lineplot(data=tcm, x='k', y='calinski_harabasz', marker='o', ax=ax_c)
    ax_c.axvline(best_k, linestyle='--', linewidth=1, color='black')
    ax_c.set_title('Calinski-Harabasz (higher better)')
    ax_c.set_xlabel('k')
    ax_c.set_ylabel('Score')

if not dbscan_dbcv_metrics.empty:
    db_plot = dbscan_dbcv_metrics.copy()
    db_plot['min_samples'] = db_plot['min_samples'].astype(str)

    if HAS_DBCV and db_plot['dbcv'].notna().any():
        sns.lineplot(data=db_plot, x='eps', y='dbcv', hue='min_samples', marker='o', ax=ax_d)
        ax_d.set_title('DBCV across DBSCAN settings (higher better)')
        ax_d.set_ylabel('DBCV')
    else:
        sns.lineplot(data=db_plot, x='eps', y='silhouette_non_noise', hue='min_samples', marker='o', ax=ax_d)
        ax_d.set_title('DBSCAN silhouette (non-noise fallback)')
        ax_d.set_ylabel('Silhouette (non-noise)')

    if np.all(db_plot['eps'].to_numpy(dtype=float) > 0):
        ax_d.set_xscale('log')
    ax_d.set_xlabel('eps')
    ax_d.legend(title='min_samples', loc='best')
else:
    ax_d.text(0.5, 0.5, 'No DBSCAN diagnostic rows', ha='center', va='center')
    ax_d.set_title('DBCV / DBSCAN diagnostics')
    ax_d.set_xticks([])
    ax_d.set_yticks([])

plt.tight_layout()
plt.savefig('clustering_metrics.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: clustering_metrics.png')

_best_dbcv = float(best_dbcv_row['dbcv']) if best_dbcv_row is not None and np.isfinite(best_dbcv_row.get('dbcv', np.nan)) else np.nan
_best_eps = float(best_dbcv_row['eps']) if best_dbcv_row is not None else np.nan
_best_min_samples = int(best_dbcv_row['min_samples']) if best_dbcv_row is not None else np.nan
transformer_embedding_meta = pd.DataFrame([{
    'n_wallet_embeddings': int(len(X_emb)),
    'n_eval_embeddings': int(len(X_eval)),
    'best_k': int(best_k),
    'sampled_for_eval': bool(len(X_emb) > MAX_EVAL_N),
    'dbcv_available': bool(HAS_DBCV),
    'dbcv_best': _best_dbcv,
    'dbscan_best_eps': _best_eps,
    'dbscan_best_min_samples': _best_min_samples,
}])


In [ ]:
# ── Cluster Archetype Validation & Naming ─────────────────────────────────────
# We compute the mean reconstruction error profile for each discovered cluster
# and automatically assign semantic labels based on their top features.

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

_flagged_ws = wallet_scores[
    wallet_scores['conformal_flag'].fillna(0).astype(int) == 1]['wallet']
_flagged_tx_mask = df_flat['wallet'].isin(_flagged_ws).values

_cluster_names = [c for c in wallet_scores['semantic_anomaly_type'].unique() if c not in ('Noise', 'Unassigned') and str(c) != 'nan']
_cluster_profiles = {}

for _cluster in _cluster_names:
    _mask = (df_flat['anomaly_type'] == _cluster) & _flagged_tx_mask
    if _mask.sum() == 0:
        _cluster_profiles[_cluster] = np.zeros(len(tx_feature_cols))
    else:
        _cluster_profiles[_cluster] = per_feat_errors[_mask].mean(axis=0)

if len(_cluster_names) > 0:
    cluster_profile_df = pd.DataFrame(_cluster_profiles, index=tx_feature_cols)
    
    # --- Auto-Labeling Logic ---
    cluster_mapping = {}
    for col in cluster_profile_df.columns:
        prof = cluster_profile_df[col].sort_values(ascending=False)
        top_feats = prof.head(3).index.tolist()
        max_val = prof.max()
        
        if max_val < 0.01:
            cluster_mapping[col] = f"{col} (Minor/IF-Driven)"
        elif 'num_post_token_balances' in top_feats or 'unique_destinations' in top_feats:
            cluster_mapping[col] = f"{col} (Drainer)"
        elif 'wallet_sol_delta' in top_feats and 'num_instructions' in top_feats:
            cluster_mapping[col] = f"{col} (Smart Contract Exploit)"
        elif 'compute_units_consumed' in top_feats and 'num_instructions' not in top_feats:
            cluster_mapping[col] = f"{col} (MEV Bot)"
        elif 'num_instructions' in top_feats or 'num_writable' in top_feats:
            cluster_mapping[col] = f"{col} (Spam/Sybil)"
        else:
            cluster_mapping[col] = f"{col} (Unknown)"
            
    # Apply mapping to dataframe and local variables
    wallet_scores['semantic_anomaly_type'] = wallet_scores['semantic_anomaly_type'].map(cluster_mapping).fillna(wallet_scores['semantic_anomaly_type'])
    df_flat['semantic_anomaly_type'] = df_flat['anomaly_type'].map(cluster_mapping).fillna(df_flat['anomaly_type'])
    cluster_profile_df = cluster_profile_df.rename(columns=cluster_mapping)
    _semantic_names = list(cluster_profile_df.columns)

    print('Mean reconstruction error per discovered cluster (flagged txs only):')
    print(cluster_profile_df.round(5).to_string())

    fig, axes = plt.subplots(1, len(_semantic_names), figsize=(5 * len(_semantic_names), 5))
    if len(_semantic_names) == 1:
        axes = [axes]
    
    for ax, _cluster in zip(axes, _semantic_names):
        _prof = cluster_profile_df[_cluster].nlargest(10).sort_values(ascending=False)
        _plot_df = pd.DataFrame({'feature': _prof.index, 'mean_recon_error': _prof.values})
        sns.barplot(data=_plot_df, x='mean_recon_error', y='feature', ax=ax, color='#F58518')
        ax.set_title(f'{_cluster}\n(top-10 features)', fontsize=9)
        ax.set_xlabel('Mean recon error')
        ax.set_ylabel('Feature')
    
    plt.suptitle('Data-Driven Cluster Profiles (flagged transactions)', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('cluster_validation.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    print('Saved: cluster_validation.png')
else:
    print("No valid clusters to plot.")


In [ ]:
# ── Final output ──────────────────────────────────────────────────────────────
df_flat['if_tx_score']  = if_tx_score
df_flat['tr_tx_score']  = tr_tx_score
df_flat['anomaly_type'] = tx_anomaly_type

wallet_scores['risk_score'] = -np.log10(
    np.clip(wallet_scores['conformal_p'].fillna(1.0), 1e-12, 1.0))
wallet_scores['risk_tier'] = pd.cut(
    wallet_scores['risk_score'],
    bins=[-np.inf, 1, 2, 3, np.inf],
    labels=['low', 'medium', 'high', 'critical'])

print('Wallet scores shape :', wallet_scores.shape)
print('df_flat shape       :', df_flat.shape)
print('\nTop 20 wallets by raw_score:')
show = ['wallet','raw_score','if_score','tr_score','semantic_anomaly_type',
        'conformal_p','conformal_flag','risk_tier','n_tx']
show = [c for c in show if c in wallet_scores.columns]
print(wallet_scores.nlargest(20, 'raw_score')[show].to_string(index=False))

# ── Optional: save ────────────────────────────────────────────────────────────
# wallet_scores.to_parquet('/home/ubuntu/data/chunks/wallet_scores.parquet', index=False)
# df_flat.to_parquet('/home/ubuntu/data/chunks/transactions_scored.parquet', index=False)

In [ ]:
# ── External baselines: LOF, One-Class SVM, MLP Autoencoder ──────────────────
# Compared to the baseline (IF + Transformer AE) via wallet-level Jaccard
# and rank correlation on the test set.  No labels required.

from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm       import OneClassSVM

# ── LOF ──────────────────────────────────────────────────────────────────────
print('Fitting LOF ...')
_lof = LocalOutlierFactor(n_neighbors=20, contamination='auto',
                          novelty=True, n_jobs=-1)
_rng_lof = np.random.default_rng(42)
_lof_n = train_mask_tx.sum()
_lof_idx = _rng_lof.choice(_lof_n, size=min(25000, _lof_n), replace=False)
_lof.fit(X_tx_norm[train_mask_tx][_lof_idx])
_lof_raw   = -_lof.decision_function(X_tx_norm)
_lof_score = (_lof_raw - _lof_raw.min()) / (_lof_raw.max() - _lof_raw.min())
lof_wallet_score = (pd.Series(_lof_score, index=df_flat.index)
                    .groupby(df_flat['wallet']).max()
                    .rename('lof_score'))

# ── One-Class SVM (subsample — OCSVM doesn't scale to 500 k rows) ────────────
print('Fitting OCSVM (15 k train sample) ...')
_rng_oc  = np.random.default_rng(42)
_oc_n    = train_mask_tx.sum()
_oc_idx  = _rng_oc.choice(_oc_n, size=min(15000, _oc_n), replace=False)
_ocsvm   = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
_ocsvm.fit(X_tx_norm[train_mask_tx][_oc_idx])
_ocsvm_raw   = -_ocsvm.decision_function(X_tx_norm)
_ocsvm_score = (_ocsvm_raw - _ocsvm_raw.min()) / (_ocsvm_raw.max() - _ocsvm_raw.min())
ocsvm_wallet_score = (pd.Series(_ocsvm_score, index=df_flat.index)
                      .groupby(df_flat['wallet']).max()
                      .rename('ocsvm_score'))

# ── MLP Autoencoder ───────────────────────────────────────────────────────────
print('Fitting MLP AE ...')
class MlpAE(nn.Module):
    def __init__(self, n_feat, hidden=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_feat, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 32),     nn.ReLU())
        self.dec = nn.Sequential(nn.Linear(32, hidden),     nn.ReLU(),
                                 nn.Linear(hidden, n_feat))
    def forward(self, x):
        return self.dec(self.enc(x))

_mlp = MlpAE(X_tx_norm.shape[1]).to(device)
_mlp_opt = torch.optim.Adam(_mlp.parameters(), lr=1e-3)
_mlp_dl  = DataLoader(TensorDataset(torch.from_numpy(X_tx_norm[train_mask_tx])),
                      batch_size=256, shuffle=True)
_mlp.train()
for _ep in range(6):
    _tot = 0
    for (_xb,) in _mlp_dl:
        _xb  = _xb.to(device)
        _l   = nn.functional.mse_loss(_mlp(_xb), _xb)
        _mlp_opt.zero_grad(); _l.backward(); _mlp_opt.step()
        _tot += _l.item()
    print(f'  MLP AE epoch {_ep+1}/6  loss={_tot/len(_mlp_dl):.6f}')

_mlp.eval()
_mlp_err = []
with torch.no_grad():
    for _i in range(0, len(X_tx_norm), 1024):
        _xb = torch.from_numpy(X_tx_norm[_i:_i+1024]).to(device)
        _mlp_err.append(((_mlp(_xb) - _xb)**2).mean(dim=1).cpu().numpy())
_mlp_raw   = np.concatenate(_mlp_err)
_mlp_score = (_mlp_raw - _mlp_raw.min()) / (_mlp_raw.max() - _mlp_raw.min())
mlp_wallet_score = (pd.Series(_mlp_score, index=df_flat.index)
                    .groupby(df_flat['wallet']).max()
                    .rename('mlp_score'))

# ── Wallet-level comparison ───────────────────────────────────────────────────
_comp_ws = (wallet_scores[['wallet', 'raw_score']]
            .merge(lof_wallet_score.reset_index(),   on='wallet', how='left')
            .merge(ocsvm_wallet_score.reset_index(), on='wallet', how='left')
            .merge(mlp_wallet_score.reset_index(),   on='wallet', how='left'))
_comp_ws = _comp_ws.rename(columns={'raw_score': 'baseline'})

_val_m  = _comp_ws['wallet'].isin(val_wallets).values
_test_m = _comp_ws['wallet'].isin(test_wallets).values

_base_cal  = _comp_ws.loc[_val_m,  'baseline'].values
_base_test = _comp_ws.loc[_test_m, 'baseline'].values
_base_fl   = (_base_test >= np.quantile(_base_cal, 1 - ALPHA)).astype(int)

_ext_rows = []
for _det in ['baseline', 'lof_score', 'ocsvm_score', 'mlp_score']:
    _cal_s  = _comp_ws.loc[_val_m,  _det].fillna(0).values
    _test_s = _comp_ws.loc[_test_m, _det].fillna(0).values
    _fl     = (_test_s >= np.quantile(_cal_s, 1 - ALPHA)).astype(int)
    _inter  = int(np.sum((_fl == 1) & (_base_fl == 1)))
    _union  = int(np.sum((_fl == 1) | (_base_fl == 1)))
    _jac    = _inter / _union if _union > 0 else 1.0
    _rho    = float(spearmanr(_test_s, _base_test).correlation)
    _ext_rows.append({'detector': _det,
                      'flagged': int(_fl.sum()),
                      'flag_rate_pct': round(float(_fl.mean() * 100), 2),
                      'jaccard_vs_baseline': round(_jac, 4),
                      'rank_corr_vs_baseline': round(_rho, 4)})

external_baselines_df = pd.DataFrame(_ext_rows)
print('\nExternal baseline comparison (wallet-level):')
print(external_baselines_df.to_string(index=False))
print('\nInterpretation: high Jaccard/rank-corr with baseline → consistent signal;'
      ' low → detectors disagree on which wallets are anomalous.')


In [ ]:
# ── 2. Ablation: component contribution vs baseline ───────────────────────────

def conformal_flags(cal_scores, test_scores, alpha):
    threshold = np.quantile(cal_scores, 1 - alpha)
    pvals = np.array([
        (np.sum(cal_scores >= s) + 1) / (len(cal_scores) + 1)
        for s in test_scores
    ])
    flags = (pvals <= alpha).astype(int)
    return threshold, pvals, flags

val_mask_w  = wallet_scores['wallet'].isin(val_wallets).values
test_mask_w = wallet_scores['wallet'].isin(test_wallets).values

score_sets = {
    'baseline': wallet_scores['raw_score'].values,
    'if_only': wallet_scores['if_score'].values,
    'tr_only': wallet_scores['tr_score'].values,
}

_, _, baseline_test_flags = conformal_flags(
    score_sets['baseline'][val_mask_w],
    score_sets['baseline'][test_mask_w],
    ALPHA,
)

rows = []
for name, scores in score_sets.items():
    thr, pvals, flags = conformal_flags(scores[val_mask_w], scores[test_mask_w], ALPHA)

    inter = np.sum((flags == 1) & (baseline_test_flags == 1))
    union = np.sum((flags == 1) | (baseline_test_flags == 1))
    jaccard = inter / union if union > 0 else 1.0

    rows.append({
        'run': name,
        'threshold': float(thr),
        'flagged_wallets': int(flags.sum()),
        'flag_rate_pct': float(flags.mean() * 100),
        'jaccard_vs_baseline': float(jaccard),
    })

ablation_results = pd.DataFrame(rows)
base_rate = float(ablation_results.loc[ablation_results['run'] == 'baseline', 'flag_rate_pct'].iloc[0])
ablation_results['delta_rate_vs_baseline'] = ablation_results['flag_rate_pct'] - base_rate

print('Ablation results (wallet-level conformal flags):')
print(ablation_results.sort_values('run').to_string(index=False))

In [ ]:
# ── Bootstrap confidence intervals for ablation Jaccard ───────────────────────
# Adds 95 % CI to each ablation variant to quantify estimation uncertainty.

_rng_bs = np.random.default_rng(42)
_N_BOOT = 1000

_val_w_arr  = wallet_scores['wallet'].isin(val_wallets).values
_test_w_arr = wallet_scores['wallet'].isin(test_wallets).values

def _bootstrap_jaccard(score_arr, base_arr, val_m, test_m, alpha, n_boot, rng):
    _cal_s  = score_arr[val_m];  _tst_s  = score_arr[test_m]
    _bcal_s = base_arr[val_m];   _btst_s = base_arr[test_m]
    _jacs   = []
    for _ in range(n_boot):
        _bi  = rng.choice(len(_tst_s), size=len(_tst_s), replace=True)
        _fl  = (_tst_s[_bi]  >= np.quantile(_cal_s,  1 - alpha)).astype(int)
        _bfl = (_btst_s[_bi] >= np.quantile(_bcal_s, 1 - alpha)).astype(int)
        _i   = np.sum((_fl == 1) & (_bfl == 1))
        _u   = np.sum((_fl == 1) | (_bfl == 1))
        _jacs.append(_i / _u if _u > 0 else 1.0)
    _j = np.array(_jacs)
    return float(np.mean(_j)), float(np.percentile(_j, 2.5)), float(np.percentile(_j, 97.5))

_baseline_arr = score_sets['baseline']
bs_rows = []
for _name, _scores in score_sets.items():
    _m, _lo, _hi = _bootstrap_jaccard(_scores, _baseline_arr,
                                       _val_w_arr, _test_w_arr,
                                       ALPHA, _N_BOOT, _rng_bs)
    bs_rows.append({'run': _name,
                    'jaccard_mean': round(_m,  4),
                    'ci_2.5':       round(_lo, 4),
                    'ci_97.5':      round(_hi, 4),
                    'ci_width':     round(_hi - _lo, 4)})

bootstrap_ci_df = pd.DataFrame(bs_rows)
print(f'Bootstrap Jaccard CIs (n_boot={_N_BOOT}, 95 % CI):')
print(bootstrap_ci_df.to_string(index=False))


In [ ]:
# ── 3. Sensitivity: stability across hyperparameter ranges ───────────────────

w_if_grid = [0.30, 0.40, 0.50, 0.60, 0.70]
alpha_grid = [0.005, 0.01, 0.02]

if_all = wallet_scores['if_score'].values
tr_all = wallet_scores['tr_score'].values

baseline_test_scores = score_sets['baseline'][test_mask_w]

sens_rows = []
for w_if in w_if_grid:
    w_tr = 1.0 - w_if
    mixed_scores = w_if * if_all + w_tr * tr_all

    for alpha in alpha_grid:
        _, _, flags = conformal_flags(mixed_scores[val_mask_w], mixed_scores[test_mask_w], alpha)

        inter = np.sum((flags == 1) & (baseline_test_flags == 1))
        union = np.sum((flags == 1) | (baseline_test_flags == 1))
        jaccard = inter / union if union > 0 else 1.0

        rho = spearmanr(mixed_scores[test_mask_w], baseline_test_scores).correlation
        if np.isnan(rho):
            rho = 0.0

        sens_rows.append({
            'w_if': float(w_if),
            'w_tr': float(w_tr),
            'alpha': float(alpha),
            'flag_rate_pct': float(flags.mean() * 100),
            'jaccard_vs_baseline': float(jaccard),
            'rank_corr_vs_baseline': float(rho),
        })

sensitivity_results = pd.DataFrame(sens_rows)
print('Sensitivity sweep:')
print(sensitivity_results.sort_values(['alpha', 'w_if']).to_string(index=False))

In [ ]:
# ── Statistical significance tests ──────────────────────────────────────────
# McNemar's test: are the disagreements between baseline and each ablation
# variant asymmetric?  This is the correct paired test for binary detectors —
# it asks whether one detector consistently flags wallets the other misses,
# beyond what chance would produce.

from scipy.stats import mcnemar

_val_w_arr  = wallet_scores['wallet'].isin(val_wallets).values
_test_w_arr = wallet_scores['wallet'].isin(test_wallets).values

def _get_flags(score_arr, val_m, test_m, alpha):
    thr = np.quantile(score_arr[val_m], 1 - alpha)
    return (score_arr[test_m] >= thr).astype(int)

_base_fl = _get_flags(score_sets['baseline'], _val_w_arr, _test_w_arr, ALPHA)

print('McNemar test (baseline vs ablation, test-set wallet flags):')
print(f'  {"comparison":<28} {"chi2":>8} {"p-value":>12} {"sig?":>6}')
print('  ' + '-' * 58)
for _nm, _scores in score_sets.items():
    if _nm == 'baseline':
        continue
    _abl_fl = _get_flags(_scores, _val_w_arr, _test_w_arr, ALPHA)
    # b = baseline flags but ablation doesn't; c = ablation flags but baseline doesn't
    _b = int(np.sum((_base_fl == 1) & (_abl_fl == 0)))
    _c = int(np.sum((_base_fl == 0) & (_abl_fl == 1)))
    _res = mcnemar([[0, _b], [_c, 0]], exact=False, correction=True)
    _sig = 'YES' if _res.pvalue < 0.05 else 'no'
    print(f'  {"baseline vs " + _nm:<28} {_res.statistic:>8.3f} {_res.pvalue:>12.4e} {_sig:>6}')
    print(f'    baseline-only flags: {_b}  |  {_nm}-only flags: {_c}')

# Permutation test: is the observed Jaccard above chance?
_rng_p  = np.random.default_rng(42)
_N_PERM = 500

_base_val = score_sets['baseline'][_val_w_arr]
_base_tst = score_sets['baseline'][_test_w_arr]
_base_fl2 = (_base_tst >= np.quantile(_base_val, 1 - ALPHA)).astype(int)

def _obs_jaccard(fl_a, fl_b):
    _i = np.sum((fl_a == 1) & (fl_b == 1))
    _u = np.sum((fl_a == 1) | (fl_b == 1))
    return _i / _u if _u > 0 else 1.0

perm_rows = []
for _nm, _sc in score_sets.items():
    if _nm == 'baseline':
        continue
    _val_s  = _sc[_val_w_arr]
    _tst_s  = _sc[_test_w_arr]
    _obs_fl = (_tst_s >= np.quantile(_val_s, 1 - ALPHA)).astype(int)
    _obs_j  = _obs_jaccard(_obs_fl, _base_fl2)
    _null_j = np.empty(_N_PERM, dtype=float)
    for _b in range(_N_PERM):
        _shuf = _rng_p.permutation(_tst_s)
        _fl_s = (_shuf >= np.quantile(_val_s, 1 - ALPHA)).astype(int)
        _null_j[_b] = _obs_jaccard(_fl_s, _base_fl2)

    # Add-one correction so permutation p-values are never reported as 0.
    _k = int(np.sum(_null_j >= _obs_j))
    _pval = float((_k + 1) / (_N_PERM + 1))

    perm_rows.append({'comparison': f'baseline vs {_nm}',
                      'observed_jaccard': round(_obs_j, 4),
                      'null_mean_jaccard': round(float(_null_j.mean()), 4),
                      'k_ge_observed': _k,
                      'perm_p': round(_pval, 4),
                      'significant': _pval < 0.05})

perm_df = pd.DataFrame(perm_rows)
print(f'\nPermutation test for Jaccard (n_perm={_N_PERM}):')
print(perm_df.to_string(index=False))
print('\nPermutation p-value uses add-one correction: p = (k + 1) / (n_perm + 1).')


In [ ]:
# ── Full-feature permutation null: shuffle features + retrain ────────────────
# This is a stronger no-label test than simple flag overlap. We destroy
# cross-feature structure, retrain baseline components, and compare anomaly
# contrast against the original model.

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

if 'score_sets' not in globals() or 'wallet_scores' not in globals():
    raise RuntimeError('Run baseline + ablation cells first (`score_sets` / `wallet_scores` missing).')

_struct_rng = np.random.default_rng(42)
N_STRUCT_PERM = 20
STRUCT_PERM_AE_EPOCHS = int(min(TR_CFG.get('epochs', 10), 4)) if 'TR_CFG' in globals() else 4

_orig_val_mask = wallet_scores['wallet'].isin(val_wallets).to_numpy()
_orig_test_mask = wallet_scores['wallet'].isin(test_wallets).to_numpy()

_orig_val_scores = score_sets['baseline'][_orig_val_mask]
_orig_test_scores = score_sets['baseline'][_orig_test_mask]
_orig_flags = (_orig_test_scores >= np.quantile(_orig_val_scores, 1 - ALPHA)).astype(int)

# Primary statistic: how separated the anomaly tail is from the center.
_orig_tail_contrast = float(np.quantile(_orig_test_scores, 0.99) - np.quantile(_orig_test_scores, 0.50))


def _jaccard_local(a, b):
    i = np.sum((a == 1) & (b == 1))
    u = np.sum((a == 1) | (b == 1))
    return float(i / u) if u > 0 else 1.0


_struct_rows = []
_struct_tail_null = []

for _b in range(N_STRUCT_PERM):
    print(f'Feature-permutation run {_b + 1}/{N_STRUCT_PERM} ...')

    # 1) Shuffle each feature independently (preserves marginals, breaks structure)
    _X_perm = X_tx.copy()
    for _j in range(_X_perm.shape[1]):
        _X_perm[:, _j] = _struct_rng.permutation(_X_perm[:, _j])

    # 2) Refit IF on shuffled data
    _if_model = IsolationForest(
        n_estimators=int(IF_CFG.get('n_estimators', 300)) if 'IF_CFG' in globals() else 300,
        max_samples=IF_CFG.get('max_samples', 256) if 'IF_CFG' in globals() else 256,
        contamination=IF_CFG.get('contamination', 0.005) if 'IF_CFG' in globals() else 0.005,
        random_state=42,
        n_jobs=-1,
    )
    _if_model.fit(_X_perm[train_mask_tx])
    _if_raw = -_if_model.decision_function(_X_perm)
    _if_sc = (_if_raw - _if_raw.min()) / (_if_raw.max() - _if_raw.min() + 1e-12)

    # 3) Refit Transformer AE on shuffled, normalized data
    _scaler_p = StandardScaler().fit(_X_perm[train_mask_tx])
    _X_perm_norm = _scaler_p.transform(_X_perm).astype(np.float32)
    _X_perm_norm = np.clip(_X_perm_norm, -20.0, 20.0)

    _perm_cfg = dict(TR_CFG) if 'TR_CFG' in globals() else {}
    _perm_cfg['epochs'] = STRUCT_PERM_AE_EPOCHS

    _tr_model_p, _, _ = train_tx_ae(
        _X_perm_norm[train_mask_tx],
        _X_perm_norm[val_mask_tx],
        _perm_cfg,
        device,
    )
    _tr_raw, _ = score_tx_ae(_tr_model_p, _X_perm_norm, device)
    _tr_sc = (_tr_raw - _tr_raw.min()) / (_tr_raw.max() - _tr_raw.min() + 1e-12)

    # 4) Baseline combination + wallet aggregation
    _comb_tx = W_IF * _if_sc + W_TR * _tr_sc
    _perm_wallet = (
        pd.DataFrame({'wallet': df_flat['wallet'].values, 'score': _comb_tx})
        .groupby('wallet')['score']
        .max()
        .reindex(wallet_scores['wallet'])
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    _perm_val = _perm_wallet[_orig_val_mask]
    _perm_test = _perm_wallet[_orig_test_mask]

    _perm_flags = (_perm_test >= np.quantile(_perm_val, 1 - ALPHA)).astype(int)
    _perm_jacc = _jaccard_local(_perm_flags, _orig_flags)
    _perm_tail = float(np.quantile(_perm_test, 0.99) - np.quantile(_perm_test, 0.50))

    _struct_tail_null.append(_perm_tail)
    _struct_rows.append({
        'perm_run': _b + 1,
        'tail_contrast': _perm_tail,
        'jaccard_vs_original_flags': _perm_jacc,
        'flag_rate_pct': float(_perm_flags.mean() * 100),
    })

struct_perm_df = pd.DataFrame(_struct_rows)

_k_struct = int(np.sum(np.array(_struct_tail_null) >= _orig_tail_contrast))
_p_struct = float((_k_struct + 1) / (N_STRUCT_PERM + 1))

struct_perm_summary = pd.DataFrame([{
    'original_tail_contrast': round(_orig_tail_contrast, 4),
    'null_tail_contrast_mean': round(float(np.mean(_struct_tail_null)), 4),
    'null_tail_contrast_std': round(float(np.std(_struct_tail_null)), 4),
    'k_null_ge_original': _k_struct,
    'perm_p': round(_p_struct, 4),
    'significant': _p_struct < 0.05,
}])

print('\nFeature-permutation null results:')
print(struct_perm_summary.to_string(index=False))
print('\nPer-run diagnostics:')
print(struct_perm_df.to_string(index=False))
print('\nInterpretation: significant means the original model has stronger anomaly-tail\n'
      'structure than shuffled-feature retrains, supporting non-random latent signal.')

In [ ]:
# ── 4. Contamination: baseline detection on synthetic anomalies ──────────────

rng = np.random.default_rng(42)
feat_idx = {f: i for i, f in enumerate(tx_feature_cols)}

normal_pool = np.where(test_mask_tx & (combined_tx <= np.quantile(combined_tx[test_mask_tx], 0.50)))[0]
if len(normal_pool) == 0:
    normal_pool = np.where(test_mask_tx)[0]

n_syn = int(min(5000, len(normal_pool)))
if n_syn == 0:
    raise RuntimeError('No test transactions available for contamination analysis.')
syn_pick = rng.choice(normal_pool, size=n_syn, replace=False)
X_syn_raw = X_tx[syn_pick].copy()

# Inject synthetic anomalous patterns
for f in ['fee_sol', 'compute_units_consumed', 'fanout_ratio', 'cpi_ratio', 'drain_sol_ratio', 'inner_instructions']:
    if f in feat_idx:
        j = feat_idx[f]
        X_syn_raw[:, j] = X_syn_raw[:, j] * rng.uniform(2.5, 6.0, size=n_syn)

if 'success_flag' in feat_idx:
    X_syn_raw[:, feat_idx['success_flag']] = 0.0

train_std = X_tx[train_mask_tx].std(axis=0) + 1e-6
X_syn_raw += rng.normal(0.0, 1.25 * train_std, size=X_syn_raw.shape)

# Re-score synthetic tx with trained baseline components
if_syn_raw = -iforest.decision_function(X_syn_raw)
if_syn = (if_syn_raw - if_raw.min()) / (if_raw.max() - if_raw.min() + 1e-12)

X_syn_norm = scaler.transform(X_syn_raw).astype(np.float32)
X_syn_norm = np.clip(X_syn_norm, -20.0, 20.0)
tr_syn_raw, _ = score_tx_ae(tr_model, X_syn_norm, device)
tr_syn = (tr_syn_raw - tr_raw.min()) / (tr_raw.max() - tr_raw.min() + 1e-12)

combined_syn = W_IF * if_syn + W_TR * tr_syn

if_thr = np.quantile(if_tx_score[val_mask_tx], 1 - ALPHA)
tr_thr = np.quantile(tr_tx_score[val_mask_tx], 1 - ALPHA)
comb_thr = np.quantile(combined_tx[val_mask_tx], 1 - ALPHA)

contamination_results = pd.DataFrame([
    {'detector': 'if_only', 'threshold': float(if_thr), 'synthetic_detection_rate_pct': float((if_syn >= if_thr).mean() * 100)},
    {'detector': 'tr_only', 'threshold': float(tr_thr), 'synthetic_detection_rate_pct': float((tr_syn >= tr_thr).mean() * 100)},
    {'detector': 'baseline_combined', 'threshold': float(comb_thr), 'synthetic_detection_rate_pct': float((combined_syn >= comb_thr).mean() * 100)},
])

print('Synthetic contamination detection rates:')
print(contamination_results.to_string(index=False))

In [ ]:
# ── 4b. Isolation Forest contamination sensitivity ───────────────────────────
from sklearn.ensemble import IsolationForest
from scipy.stats import spearmanr


def _conformal_flags_local(cal_scores, test_scores, alpha):
    cal = np.asarray(cal_scores, dtype=float)
    tst = np.asarray(test_scores, dtype=float)
    cal = cal[np.isfinite(cal)]
    tst = tst[np.isfinite(tst)]

    if len(cal) == 0 or len(tst) == 0:
        return np.nan, np.array([], dtype=float), np.array([], dtype=int)

    thr = float(np.quantile(cal, 1 - alpha))
    pvals = np.array([(np.sum(cal >= s) + 1) / (len(cal) + 1) for s in tst], dtype=float)
    flags = (pvals <= alpha).astype(int)
    return thr, pvals, flags


if 'wallet_scores' not in globals():
    raise RuntimeError('Run baseline scoring sections first (`wallet_scores` is missing).')

if 'ALPHA' not in globals():
    ALPHA = 0.01

if 'val_wallets' in globals():
    val_mask_w_sweep = wallet_scores['wallet'].isin(val_wallets).to_numpy()
else:
    val_mask_w_sweep = np.zeros(len(wallet_scores), dtype=bool)

if 'test_wallets' in globals():
    test_mask_w_sweep = wallet_scores['wallet'].isin(test_wallets).to_numpy()
else:
    test_mask_w_sweep = ~val_mask_w_sweep

if val_mask_w_sweep.sum() == 0:
    val_mask_w_sweep = ~test_mask_w_sweep
if test_mask_w_sweep.sum() == 0:
    test_mask_w_sweep = np.ones(len(wallet_scores), dtype=bool)

# Small sweep model for speed; increase if you want tighter estimates.
contam_grid = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.05]
sweep_n_estimators = int(min(IF_CFG.get('n_estimators', 300), 150)) if 'IF_CFG' in globals() else 150
sweep_max_samples = IF_CFG.get('max_samples', 256) if 'IF_CFG' in globals() else 256

wallet_order = wallet_scores['wallet'].to_numpy()

rows = []
test_scores_by_c = {}
test_flags_by_c = {}
wallet_scores_by_c = {}

for c_raw in contam_grid:
    c = float(c_raw)
    print(f'Fitting IF sweep contamination={c} ...')

    if_model = IsolationForest(
        n_estimators=sweep_n_estimators,
        max_samples=sweep_max_samples,
        contamination=c,
        random_state=42,
        n_jobs=-1,
    )
    if_model.fit(X_tx[train_mask_tx])

    tx_raw = -if_model.decision_function(X_tx)
    tx_norm = (tx_raw - tx_raw.min()) / (tx_raw.max() - tx_raw.min() + 1e-12)

    w_if = (
        pd.DataFrame({'wallet': df_flat['wallet'].values, 'if_score_c': tx_norm})
        .groupby('wallet')['if_score_c'].max()
        .reindex(wallet_order)
        .to_numpy(dtype=float)
    )

    wallet_scores_by_c[c] = w_if
    score_test = w_if[test_mask_w_sweep]
    score_val = w_if[val_mask_w_sweep]

    thr, _, flags = _conformal_flags_local(score_val, score_test, ALPHA)
    test_scores_by_c[c] = score_test
    test_flags_by_c[c] = flags

    rows.append({
        'contamination': c,
        'threshold': thr,
        'flagged_wallets': int(flags.sum()),
        'flag_rate_pct': float(flags.mean() * 100) if len(flags) else np.nan,
        'score_p95_test': float(np.quantile(score_test, 0.95)) if len(score_test) else np.nan,
        'score_p99_test': float(np.quantile(score_test, 0.99)) if len(score_test) else np.nan,
    })

if_sensitivity_results = pd.DataFrame(rows).sort_values('contamination').reset_index(drop=True)

# Stability against a reference contamination (0.01 by default)
ref_c = 0.01 if 0.01 in [float(x) for x in contam_grid] else float(if_sensitivity_results['contamination'].iloc[len(if_sensitivity_results) // 2])
ref_flags = test_flags_by_c.get(ref_c, np.array([], dtype=int))
ref_scores = test_scores_by_c.get(ref_c, np.array([], dtype=float))

jaccard_vs_ref = []
rank_corr_vs_ref = []
adjacent_jaccard = [np.nan]

sorted_cs = [float(x) for x in if_sensitivity_results['contamination'].to_list()]
for i, c in enumerate(sorted_cs):
    flags = test_flags_by_c[c]
    scores = test_scores_by_c[c]

    if len(ref_flags) and len(flags):
        inter = np.sum((flags == 1) & (ref_flags == 1))
        union = np.sum((flags == 1) | (ref_flags == 1))
        jac = float(inter / union) if union > 0 else 1.0
    else:
        jac = np.nan
    jaccard_vs_ref.append(jac)

    if len(ref_scores) and len(scores):
        rho = spearmanr(scores, ref_scores).correlation
        rho = float(0.0 if pd.isna(rho) else rho)
    else:
        rho = np.nan
    rank_corr_vs_ref.append(rho)

    if i > 0:
        prev_c = sorted_cs[i - 1]
        prev_flags = test_flags_by_c[prev_c]
        inter = np.sum((flags == 1) & (prev_flags == 1))
        union = np.sum((flags == 1) | (prev_flags == 1))
        adjacent_jaccard.append(float(inter / union) if union > 0 else 1.0)

if_sensitivity_results['jaccard_vs_ref'] = jaccard_vs_ref
if_sensitivity_results['rank_corr_vs_ref'] = rank_corr_vs_ref
if_sensitivity_results['jaccard_vs_prev'] = adjacent_jaccard

print(f'\nIF contamination sensitivity (reference contamination={ref_c}):')
print(if_sensitivity_results.to_string(index=False))

# Seaborn plot: score distribution + stability curve
ref_wallet_scores = wallet_scores_by_c.get(ref_c)
if ref_wallet_scores is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    _dist_df = pd.DataFrame({
        'score': np.concatenate([ref_wallet_scores[val_mask_w_sweep], ref_wallet_scores[test_mask_w_sweep]]),
        'split': (['val'] * int(np.sum(val_mask_w_sweep))) + (['test'] * int(np.sum(test_mask_w_sweep))),
    })
    sns.histplot(data=_dist_df, x='score', hue='split', bins=40, element='step', fill=False, stat='count', common_norm=False, ax=axes[0])
    axes[0].set_title(f'IF score distribution (contam={ref_c})')
    axes[0].set_xlabel('Wallet IF score')
    axes[0].set_ylabel('Count')

    _stab_df = if_sensitivity_results[['contamination', 'flag_rate_pct', 'jaccard_vs_ref', 'rank_corr_vs_ref']].copy()
    _stab_df['jaccard_vs_ref'] = 100.0 * _stab_df['jaccard_vs_ref']
    _stab_df['rank_corr_vs_ref'] = 100.0 * _stab_df['rank_corr_vs_ref']
    _stab_df = _stab_df.melt(
        id_vars='contamination',
        value_vars=['flag_rate_pct', 'jaccard_vs_ref', 'rank_corr_vs_ref'],
        var_name='metric',
        value_name='percent',
    )
    sns.lineplot(data=_stab_df, x='contamination', y='percent', hue='metric', marker='o', ax=axes[1])
    axes[1].set_xscale('log')
    axes[1].set_title('Sensitivity stability vs contamination')
    axes[1].set_xlabel('IsolationForest contamination')
    axes[1].set_ylabel('Percent')

    plt.tight_layout()
    plt.show()

if 'no_label_dashboard' in globals():
    no_label_dashboard['if_contamination_sensitivity'] = if_sensitivity_results

print("Saved table: `if_sensitivity_results` (and `no_label_dashboard['if_contamination_sensitivity']` if dashboard exists).")

In [ ]:
# ── 5. Interpretability: what baseline flagged and why ───────────────────────

tx_top_idx = df_flat.groupby('wallet')['combined_tx_score'].idxmax()
tx_top_map = tx_top_idx.to_dict()

flagged_wallets = wallet_scores[
    wallet_scores['wallet'].isin(test_wallets)
    & (wallet_scores['conformal_flag'].fillna(0).astype(int) == 1)
].copy()

if len(flagged_wallets) == 0:
    print('No flagged test wallets at current alpha.')
else:
    explain_rows = []
    for _, row in flagged_wallets.sort_values('raw_score', ascending=False).head(25).iterrows():
        w = row['wallet']
        tx_idx = tx_top_map.get(w)
        if tx_idx is None:
            continue

        feat_err = per_feat_errors[int(tx_idx)]
        topk = np.argsort(feat_err)[-5:][::-1]
        top_feats = [f"{tx_feature_cols[i]}={feat_err[i]:.4f}" for i in topk]

        dominant = 'if' if row['if_score'] >= row['tr_score'] else 'tr'

        explain_rows.append({
            'wallet': w,
            'raw_score': float(row['raw_score']),
            'if_score': float(row['if_score']),
            'tr_score': float(row['tr_score']),
            'conformal_p': float(row['conformal_p']) if pd.notna(row['conformal_p']) else np.nan,
            'semantic_anomaly_type': row.get('semantic_anomaly_type', 'unknown'),
            'dominant_component': dominant,
            'top_feature_contributors': '; '.join(top_feats),
        })

    interpretability_results = pd.DataFrame(explain_rows)
    print('Flagged-wallet explanations:')
    print(interpretability_results.to_string(index=False))

In [ ]:
# ── SHAP interpretability (Isolation Forest component) ────────────────────────
# Addresses the black-box concern by explaining which features drive IF anomaly
# scores for top anomalous test transactions.

shap_global_df = pd.DataFrame()
shap_local_top_df = pd.DataFrame()

try:
    import shap
except Exception:
    print('SHAP is not installed in this environment. Run `pip install shap` and re-run this cell.')
else:
    required_vars = ['iforest', 'if_tx_score', 'X_tx_norm', 'test_mask_tx', 'tx_feature_cols', 'df_flat']
    missing_vars = [v for v in required_vars if v not in globals()]
    if missing_vars:
        raise RuntimeError(f'Missing prerequisites for SHAP: {missing_vars}')

    bg_n = int(min(2000, np.sum(train_mask_tx))) if 'train_mask_tx' in globals() else int(min(2000, len(X_tx_norm)))
    ex_n = int(min(300, np.sum(test_mask_tx)))
    if ex_n <= 0:
        raise RuntimeError('No test transactions available for SHAP interpretation.')

    rng_shap = np.random.default_rng(42)
    if 'train_mask_tx' in globals() and np.sum(train_mask_tx) > 0:
        train_idx = np.where(train_mask_tx)[0]
    else:
        train_idx = np.arange(len(X_tx_norm))

    bg_idx = rng_shap.choice(train_idx, size=bg_n, replace=False) if len(train_idx) > bg_n else train_idx

    test_idx = np.where(test_mask_tx)[0]
    top_order = np.argsort(if_tx_score[test_mask_tx])[-ex_n:][::-1]
    explain_idx = test_idx[top_order]

    X_bg = X_tx_norm[bg_idx].astype(np.float32)
    X_ex = X_tx_norm[explain_idx].astype(np.float32)

    shap_values = None
    shap_mode = 'tree'

    try:
        explainer = shap.TreeExplainer(iforest)
        shap_values = explainer.shap_values(X_ex)
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
        shap_values = np.asarray(shap_values, dtype=float)
    except Exception:
        shap_mode = 'model_agnostic'

        def _if_anom(x):
            x = np.asarray(x, dtype=np.float32)
            return -iforest.decision_function(x)

        explainer = shap.Explainer(_if_anom, X_bg, feature_names=tx_feature_cols)
        shap_out = explainer(X_ex)
        shap_values = np.asarray(shap_out.values, dtype=float)

    if shap_values.ndim != 2 or shap_values.shape[1] != len(tx_feature_cols):
        raise RuntimeError(f'Unexpected SHAP output shape: {shap_values.shape}')

    # Global IF feature importance
    mean_abs = np.mean(np.abs(shap_values), axis=0)
    shap_global_df = (
        pd.DataFrame({'feature': tx_feature_cols, 'mean_abs_shap': mean_abs})
        .sort_values('mean_abs_shap', ascending=False)
        .reset_index(drop=True)
    )

    print(f'SHAP mode: {shap_mode}')
    print('Top IF feature drivers by mean |SHAP|:')
    print(shap_global_df.head(15).to_string(index=False))

    # Local explanations for top anomalous transactions
    top_feat = shap_global_df.head(12)['feature'].tolist()
    top_feat_idx = [tx_feature_cols.index(f) for f in top_feat]

    shap_local_top_df = pd.DataFrame(
        shap_values[:, top_feat_idx],
        columns=top_feat,
    )
    shap_local_top_df['tx_idx'] = explain_idx
    shap_local_top_df['wallet'] = df_flat.iloc[explain_idx]['wallet'].astype(str).values
    shap_local_top_df['if_score'] = if_tx_score[explain_idx]

    # Seaborn visualisations
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

    sns.barplot(data=shap_global_df.head(15), x='mean_abs_shap', y='feature', color='#4C78A8', ax=axes[0])
    axes[0].set_title('SHAP global importance (Isolation Forest)')
    axes[0].set_xlabel('Mean |SHAP value|')
    axes[0].set_ylabel('Feature')

    hm_n = int(min(20, len(shap_local_top_df)))
    hm = shap_local_top_df[top_feat].head(hm_n).to_numpy(dtype=float)
    row_labels = []
    for w in shap_local_top_df['wallet'].head(hm_n):
        ws = str(w)
        row_labels.append(ws if len(ws) <= 14 else f'{ws[:6]}...{ws[-4:]}')

    sns.heatmap(
        hm,
        cmap='coolwarm',
        center=0.0,
        xticklabels=top_feat,
        yticklabels=row_labels,
        cbar_kws={'label': 'SHAP value'},
        ax=axes[1],
    )
    axes[1].set_title('Local SHAP (top anomalous test transactions)')
    axes[1].set_xlabel('Feature')
    axes[1].set_ylabel('Transaction (wallet)')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    if 'no_label_dashboard' in globals():
        no_label_dashboard['if_shap_global'] = shap_global_df
        no_label_dashboard['if_shap_local_top'] = shap_local_top_df

print('Saved SHAP outputs: `shap_global_df`, `shap_local_top_df` (and dashboard keys if available).')

In [ ]:
# ── 6. No-label evaluation dashboard ──────────────────────────────────────────
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.metrics import roc_auc_score, average_precision_score


def _clean(x):
    x = np.asarray(x, dtype=float)
    return x[np.isfinite(x)]


def _score_summary(name, values):
    x = _clean(values)
    if len(x) == 0:
        return {
            'detector': name,
            'n': 0,
            'mean': np.nan,
            'std': np.nan,
            'p50': np.nan,
            'p90': np.nan,
            'p95': np.nan,
            'p99': np.nan,
            'max': np.nan,
        }
    return {
        'detector': name,
        'n': int(len(x)),
        'mean': float(np.mean(x)),
        'std': float(np.std(x)),
        'p50': float(np.quantile(x, 0.50)),
        'p90': float(np.quantile(x, 0.90)),
        'p95': float(np.quantile(x, 0.95)),
        'p99': float(np.quantile(x, 0.99)),
        'max': float(np.max(x)),
    }


def _psi(expected, actual, bins=10, eps=1e-6):
    exp = _clean(expected)
    act = _clean(actual)
    if len(exp) == 0 or len(act) == 0:
        return np.nan

    edges = np.quantile(exp, np.linspace(0, 1, bins + 1))
    edges = np.unique(edges)
    if len(edges) < 3:
        return 0.0

    exp_hist, _ = np.histogram(exp, bins=edges)
    act_hist, _ = np.histogram(act, bins=edges)

    exp_pct = np.clip(exp_hist / max(exp_hist.sum(), 1), eps, None)
    act_pct = np.clip(act_hist / max(act_hist.sum(), 1), eps, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def _topk_jaccard(a, b, k):
    a = _clean(a)
    b = _clean(b)
    k = int(min(k, len(a), len(b)))
    if k <= 0:
        return np.nan

    set_a = set(np.argsort(a)[-k:])
    set_b = set(np.argsort(b)[-k:])
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return float(inter / union) if union else 1.0


def _conformal_rate(cal_scores, test_scores, alpha):
    cal = _clean(cal_scores)
    tst = _clean(test_scores)
    if len(cal) == 0 or len(tst) == 0:
        return np.nan, np.nan

    thr = float(np.quantile(cal, 1 - alpha))
    pvals = np.array([(np.sum(cal >= s) + 1) / (len(cal) + 1) for s in tst])
    rate = float((pvals <= alpha).mean())
    return thr, rate


def _drift_row(name, ref, cur):
    ref = _clean(ref)
    cur = _clean(cur)
    if len(ref) == 0 or len(cur) == 0:
        return {
            'score': name,
            'ks_stat': np.nan,
            'ks_pvalue': np.nan,
            'wasserstein': np.nan,
            'psi_10bins': np.nan,
        }

    ks_stat, ks_p = ks_2samp(ref, cur)
    return {
        'score': name,
        'ks_stat': float(ks_stat),
        'ks_pvalue': float(ks_p),
        'wasserstein': float(wasserstein_distance(ref, cur)),
        'psi_10bins': float(_psi(ref, cur, bins=10)),
    }


required = {'wallet', 'if_score', 'tr_score', 'raw_score'}
missing = sorted(required - set(wallet_scores.columns))
if missing:
    raise ValueError(f'Missing required wallet score columns: {missing}')

# Evaluation masks (prefer true temporal test/val wallet split if available)
if 'test_wallets' in globals():
    test_mask_eval = wallet_scores['wallet'].isin(test_wallets).values
else:
    test_mask_eval = np.ones(len(wallet_scores), dtype=bool)

if 'val_wallets' in globals():
    val_mask_eval = wallet_scores['wallet'].isin(val_wallets).values
else:
    val_mask_eval = ~test_mask_eval

if val_mask_eval.sum() == 0:
    val_mask_eval = test_mask_eval.copy()

# Core vectors
if_test = wallet_scores.loc[test_mask_eval, 'if_score'].to_numpy(dtype=float)
tr_test = wallet_scores.loc[test_mask_eval, 'tr_score'].to_numpy(dtype=float)
raw_test = wallet_scores.loc[test_mask_eval, 'raw_score'].to_numpy(dtype=float)

if_val = wallet_scores.loc[val_mask_eval, 'if_score'].to_numpy(dtype=float)
tr_val = wallet_scores.loc[val_mask_eval, 'tr_score'].to_numpy(dtype=float)
raw_val = wallet_scores.loc[val_mask_eval, 'raw_score'].to_numpy(dtype=float)

if 'conformal_flag' in wallet_scores.columns:
    flags_test = wallet_scores.loc[test_mask_eval, 'conformal_flag'].fillna(0).astype(int).to_numpy()
else:
    flags_test = np.zeros(test_mask_eval.sum(), dtype=int)

if 'conformal_p' in wallet_scores.columns:
    pvals_test = wallet_scores.loc[test_mask_eval, 'conformal_p'].to_numpy(dtype=float)
else:
    pvals_test = np.array([], dtype=float)

# 1) Workload / alert-rate view
n_test = int(test_mask_eval.sum())
n_flagged = int(flags_test.sum())
flag_rate = float(n_flagged / max(n_test, 1))

no_label_overview = pd.DataFrame([{
    'n_test_wallets': n_test,
    'n_flagged_wallets': n_flagged,
    'flag_rate_pct': flag_rate * 100,
    'median_risk_score': float(wallet_scores.loc[test_mask_eval, 'risk_score'].median()) if 'risk_score' in wallet_scores.columns else np.nan,
    'p95_risk_score': float(wallet_scores.loc[test_mask_eval, 'risk_score'].quantile(0.95)) if 'risk_score' in wallet_scores.columns else np.nan,
    'conformal_p_median': float(np.nanmedian(pvals_test)) if len(pvals_test) else np.nan,
    'conformal_p_p05': float(np.nanquantile(pvals_test, 0.05)) if len(pvals_test) else np.nan,
}])

print('No-label overview:')
print(no_label_overview.to_string(index=False))

# Time-based analyst workload (daily alerts from top-scored tx per wallet)
alerts_over_time = pd.DataFrame()
if {'wallet', 'block_timestamp', 'combined_tx_score'}.issubset(df_flat.columns):
    top_tx_idx = df_flat.groupby('wallet')['combined_tx_score'].idxmax()
    ts_map = (
        df_flat.loc[top_tx_idx, ['wallet', 'block_timestamp']]
        .drop_duplicates('wallet')
        .set_index('wallet')['block_timestamp']
    )
    test_wallet_df = wallet_scores.loc[test_mask_eval, ['wallet']].copy()
    test_wallet_df['flag'] = flags_test
    test_wallet_df['top_tx_ts'] = test_wallet_df['wallet'].map(ts_map)

    flagged_ts = pd.to_datetime(test_wallet_df.loc[test_wallet_df['flag'] == 1, 'top_tx_ts'], errors='coerce')
    flagged_ts = flagged_ts.dropna()
    if len(flagged_ts):
        alerts_over_time = (
            flagged_ts.dt.floor('D').value_counts().sort_index()
            .rename_axis('date').reset_index(name='alerts')
        )
        print('\nAlert workload (daily) preview:')
        print(alerts_over_time.head(10).to_string(index=False))

# 2) Conformal diagnostics: realized flag rate vs target alpha
alpha_grid = [0.005, 0.01, 0.02, 0.05]
conformal_rows = []
for a in alpha_grid:
    thr, realized_rate = _conformal_rate(raw_val, raw_test, a)
    conformal_rows.append({
        'alpha_target': float(a),
        'threshold': thr,
        'realized_flag_rate_pct': float(realized_rate * 100) if np.isfinite(realized_rate) else np.nan,
        'abs_gap_pct_points': float(abs(realized_rate - a) * 100) if np.isfinite(realized_rate) else np.nan,
    })
conformal_diagnostics = pd.DataFrame(conformal_rows)

print('\nConformal diagnostics:')
print(conformal_diagnostics.to_string(index=False))

# 3) Score diagnostics
score_diagnostics = pd.DataFrame([
    _score_summary('if_score_test', if_test),
    _score_summary('tr_score_test', tr_test),
    _score_summary('raw_score_test', raw_test),
    _score_summary('if_score_val', if_val),
    _score_summary('tr_score_val', tr_val),
    _score_summary('raw_score_val', raw_val),
])

print('\nScore diagnostics:')
print(score_diagnostics.to_string(index=False))

# 4) Model-aligned error metrics table
model_score_map = {
    'baseline': wallet_scores['raw_score'].to_numpy(dtype=float),
    'if_only': wallet_scores['if_score'].to_numpy(dtype=float),
    'tr_only': wallet_scores['tr_score'].to_numpy(dtype=float),
}

if '_comp_ws' in globals() and isinstance(_comp_ws, pd.DataFrame):
    _ext_cols = [c for c in ['lof_score', 'ocsvm_score', 'mlp_score'] if c in _comp_ws.columns]
    if len(_ext_cols):
        _ext = (
            _comp_ws[['wallet'] + _ext_cols]
            .drop_duplicates('wallet')
            .set_index('wallet')
            .reindex(wallet_scores['wallet'])
            .fillna(0.0)
        )
        for _c in _ext_cols:
            model_score_map[_c.replace('_score', '')] = _ext[_c].to_numpy(dtype=float)

_base_val_m = _clean(model_score_map['baseline'][val_mask_eval])
_base_test_m = _clean(model_score_map['baseline'][test_mask_eval])
_base_thr_m = float(np.quantile(_base_val_m, 1 - ALPHA)) if len(_base_val_m) else np.nan
_base_flags_m = (_base_test_m >= _base_thr_m).astype(int) if np.isfinite(_base_thr_m) else np.array([], dtype=int)

_syn_rate_map = {}
if 'contamination_results' in globals() and isinstance(contamination_results, pd.DataFrame) and len(contamination_results):
    _det_map = {'baseline_combined': 'baseline', 'if_only': 'if_only', 'tr_only': 'tr_only'}
    for _, _r in contamination_results.iterrows():
        _name = _det_map.get(str(_r.get('detector', '')))
        if _name:
            _syn_rate_map[_name] = float(_r.get('synthetic_detection_rate_pct', np.nan))

_model_rows = []
for _det, _arr in model_score_map.items():
    _val_s = _clean(_arr[val_mask_eval])
    _tst_s = _clean(_arr[test_mask_eval])
    if len(_val_s) == 0 or len(_tst_s) == 0:
        continue

    _thr = float(np.quantile(_val_s, 1 - ALPHA))
    _fl = (_tst_s >= _thr).astype(int)

    _inter = int(np.sum((_fl == 1) & (_base_flags_m == 1))) if len(_base_flags_m) else 0
    _union = int(np.sum((_fl == 1) | (_base_flags_m == 1))) if len(_base_flags_m) else 0
    _jac = float(_inter / _union) if _union > 0 else 1.0

    _rho = spearmanr(_tst_s, _base_test_m).correlation if len(_tst_s) == len(_base_test_m) else np.nan
    _rho = float(0.0 if pd.isna(_rho) else _rho)

    _dr = _drift_row(_det, _val_s, _tst_s)
    _tail = float(np.quantile(_tst_s, 0.99) - np.quantile(_tst_s, 0.50))

    _model_rows.append({
        'detector': _det,
        'val_threshold_alpha': _thr,
        'test_flagged_wallets': int(_fl.sum()),
        'test_flag_rate_pct': float(_fl.mean() * 100),
        'test_tail_contrast': _tail,
        'jaccard_vs_baseline': _jac,
        'rank_corr_vs_baseline': _rho,
        'ks_stat_val_test': _dr['ks_stat'],
        'wasserstein_val_test': _dr['wasserstein'],
        'psi_10bins_val_test': _dr['psi_10bins'],
        'synthetic_detection_rate_pct': float(_syn_rate_map.get(_det, np.nan)),
    })

model_error_metrics = pd.DataFrame(_model_rows)
if len(model_error_metrics):
    _ord = {'baseline': 0, 'if_only': 1, 'tr_only': 2, 'lof': 3, 'ocsvm': 4, 'mlp': 5}
    model_error_metrics['sort_key'] = model_error_metrics['detector'].map(_ord).fillna(99)
    model_error_metrics = model_error_metrics.sort_values(['sort_key', 'detector']).drop(columns=['sort_key']).reset_index(drop=True)

print('\nModel error metrics (aligned to detectors):')
print(model_error_metrics.to_string(index=False) if len(model_error_metrics) else 'No model metrics available.')

# 4b) Logical-perspective comparison (no labels)
logical_perspective_summary = pd.DataFrame()
if len(model_error_metrics):
    logical_perspective_summary = model_error_metrics.copy()

if 'ablation_results' in globals() and isinstance(ablation_results, pd.DataFrame) and len(ablation_results):
    _abl = (
        ablation_results.rename(columns={'run': 'detector'})
        [['detector', 'jaccard_vs_baseline', 'delta_rate_vs_baseline']]
        .rename(columns={
            'jaccard_vs_baseline': 'ablation_jaccard_vs_baseline',
            'delta_rate_vs_baseline': 'ablation_delta_flag_rate_pct',
        })
    )
    logical_perspective_summary = (
        _abl if len(logical_perspective_summary) == 0
        else logical_perspective_summary.merge(_abl, on='detector', how='outer')
    )

if 'perm_df' in globals() and isinstance(perm_df, pd.DataFrame) and len(perm_df):
    _perm = perm_df.copy()
    _perm['detector'] = _perm['comparison'].astype(str).str.replace('baseline vs ', '', regex=False)
    _perm = _perm[['detector', 'observed_jaccard', 'perm_p']].rename(columns={
        'observed_jaccard': 'perm_observed_jaccard',
        'perm_p': 'perm_p_jaccard_overlap',
    })
    logical_perspective_summary = (
        _perm if len(logical_perspective_summary) == 0
        else logical_perspective_summary.merge(_perm, on='detector', how='outer')
    )

if len(logical_perspective_summary):
    _ord = {'baseline': 0, 'if_only': 1, 'tr_only': 2, 'lof': 3, 'ocsvm': 4, 'mlp': 5}
    logical_perspective_summary['sort_key'] = logical_perspective_summary['detector'].map(_ord).fillna(99)
    logical_perspective_summary = (
        logical_perspective_summary
        .sort_values(['sort_key', 'detector'])
        .drop(columns=['sort_key'])
        .reset_index(drop=True)
    )

print('\nLogical perspective summary (ablation + stability + permutation):')
print(logical_perspective_summary.to_string(index=False) if len(logical_perspective_summary) else 'No logical perspective summary available.')

# 5) Detector agreement and top-k overlap
agreement_corr = pd.DataFrame({
    'if_score': if_test,
    'tr_score': tr_test,
    'raw_score': raw_test,
}).corr(method='spearman')

overlap_rows = []
for k in [50, 100, 250]:
    overlap_rows.append({
        'k': int(k),
        'jaccard_if_vs_tr': _topk_jaccard(if_test, tr_test, k),
        'jaccard_if_vs_raw': _topk_jaccard(if_test, raw_test, k),
        'jaccard_tr_vs_raw': _topk_jaccard(tr_test, raw_test, k),
    })
detector_topk_overlap = pd.DataFrame(overlap_rows)

print('\nDetector agreement (Spearman correlation):')
print(agreement_corr.to_string())
print('\nTop-k overlap (Jaccard):')
print(detector_topk_overlap.to_string(index=False))

# 5) Drift diagnostics (val -> test)
drift_results = pd.DataFrame([
    _drift_row('if_score', if_val, if_test),
    _drift_row('tr_score', tr_val, tr_test),
    _drift_row('raw_score', raw_val, raw_test),
])

print('\nDrift diagnostics (val -> test):')
print(drift_results.to_string(index=False))

# 6) Reconstruction feature lift on flagged wallets (no labels)
feature_lift_results = pd.DataFrame()
if 'per_feat_errors' in globals() and isinstance(per_feat_errors, np.ndarray) and len(per_feat_errors) == len(df_flat):
    top_tx_by_wallet = df_flat.groupby('wallet')['combined_tx_score'].idxmax()
    test_wallet_df = wallet_scores.loc[test_mask_eval, ['wallet']].copy()
    test_wallet_df['flag'] = flags_test

    flagged_wallets = test_wallet_df.loc[test_wallet_df['flag'] == 1, 'wallet']
    unflagged_wallets = test_wallet_df.loc[test_wallet_df['flag'] == 0, 'wallet']

    flagged_idx = top_tx_by_wallet.reindex(flagged_wallets).dropna().astype(int).to_numpy()
    unflagged_idx = top_tx_by_wallet.reindex(unflagged_wallets).dropna().astype(int).to_numpy()

    if len(flagged_idx) > 0 and len(unflagged_idx) > 0:
        flagged_mean = per_feat_errors[flagged_idx].mean(axis=0)
        unflagged_mean = per_feat_errors[unflagged_idx].mean(axis=0)
        lift = (flagged_mean + 1e-12) / (unflagged_mean + 1e-12)

        feature_lift_results = pd.DataFrame({
            'feature': tx_feature_cols,
            'flagged_mean_error': flagged_mean,
            'unflagged_mean_error': unflagged_mean,
            'lift_vs_unflagged': lift,
        }).sort_values('lift_vs_unflagged', ascending=False)

        print('\nTop feature lifts (flagged vs unflagged test wallets):')
        print(feature_lift_results.head(12).to_string(index=False))
    else:
        print('\nFeature-lift view skipped: need both flagged and unflagged wallets in test set.')
else:
    print('\nFeature-lift view skipped: per-feature reconstruction errors unavailable.')

# 7) Synthetic proxy separability (if contamination cell already ran)
synthetic_proxy_metrics = pd.DataFrame()
if all(v in globals() for v in ['combined_syn', 'combined_tx', 'test_mask_tx']):
    real_ref = _clean(np.asarray(combined_tx)[np.asarray(test_mask_tx)])
    syn_ref = _clean(np.asarray(combined_syn))

    if len(real_ref) > 0 and len(syn_ref) > 0:
        y = np.concatenate([np.zeros(len(real_ref), dtype=int), np.ones(len(syn_ref), dtype=int)])
        s = np.concatenate([real_ref, syn_ref])

        try:
            roc = float(roc_auc_score(y, s))
            ap = float(average_precision_score(y, s))
        except Exception:
            roc, ap = np.nan, np.nan

        synthetic_proxy_metrics = pd.DataFrame([{
            'n_real_reference': int(len(real_ref)),
            'n_synthetic': int(len(syn_ref)),
            'synthetic_roc_auc_proxy': roc,
            'synthetic_pr_auc_proxy': ap,
        }])

        print('\nSynthetic proxy separability:')
        print(synthetic_proxy_metrics.to_string(index=False))

        if 'contamination_results' in globals():
            print('\nSynthetic threshold detection rates:')
            print(contamination_results.to_string(index=False))

# Bundle outputs for reuse
no_label_dashboard = {
    'overview': no_label_overview,
    'alerts_over_time': alerts_over_time,
    'conformal_diagnostics': conformal_diagnostics,
    'score_diagnostics': score_diagnostics,
    'model_error_metrics': model_error_metrics,
    'logical_perspective_summary': logical_perspective_summary,
    'agreement_corr': agreement_corr,
    'detector_topk_overlap': detector_topk_overlap,
    'drift_results': drift_results,
    'feature_lift_results': feature_lift_results,
    'synthetic_proxy_metrics': synthetic_proxy_metrics,
}

if 'shap_global_df' in globals() and isinstance(shap_global_df, pd.DataFrame) and len(shap_global_df):
    no_label_dashboard['if_shap_global'] = shap_global_df
if 'shap_local_top_df' in globals() and isinstance(shap_local_top_df, pd.DataFrame) and len(shap_local_top_df):
    no_label_dashboard['if_shap_local_top'] = shap_local_top_df

print('\nSaved dashboard tables in `no_label_dashboard`.')

In [ ]:
# ── 7. No-label dashboard plots (compact Seaborn renderer) ───────────────────
if 'no_label_dashboard' not in globals():
    raise RuntimeError('Run section 6 (No-label evaluation dashboard) first.')


def render_no_label_dashboard(dashboard, save_path=None, show=True, figsize=(18, 10.5)):
    alerts_over_time = dashboard.get('alerts_over_time', pd.DataFrame())
    conformal_diagnostics = dashboard.get('conformal_diagnostics', pd.DataFrame())
    drift_results = dashboard.get('drift_results', pd.DataFrame())
    feature_lift_results = dashboard.get('feature_lift_results', pd.DataFrame())
    detector_topk_overlap = dashboard.get('detector_topk_overlap', pd.DataFrame())
    model_error_metrics = dashboard.get('model_error_metrics', pd.DataFrame())

    fig, axes = plt.subplots(2, 3, figsize=figsize)
    ax1, ax2, ax3, ax4, ax5, ax6 = axes.flatten()

    def _blank(ax, title, msg):
        ax.text(0.5, 0.5, msg, ha='center', va='center')
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    # 1) Alert trend
    if isinstance(alerts_over_time, pd.DataFrame) and len(alerts_over_time):
        aot = alerts_over_time.copy()
        aot['date'] = pd.to_datetime(aot['date'], errors='coerce')
        aot = aot.dropna(subset=['date'])
        aot['alerts_roll3'] = aot['alerts'].rolling(3, min_periods=1).mean()

        sns.lineplot(data=aot, x='date', y='alerts', marker='o', linewidth=1.6, label='Daily alerts', ax=ax1)
        sns.lineplot(data=aot, x='date', y='alerts_roll3', linewidth=1.4, linestyle='--', label='3-day moving avg', ax=ax1)
        ax1.set_title('Alert workload over time')
        ax1.set_xlabel('Date')
        ax1.set_ylabel('Flagged wallets')
        ax1.legend(loc='best')
    else:
        _blank(ax1, 'Alert workload over time', 'No alert time-series available')

    # 2) Conformal calibration
    if isinstance(conformal_diagnostics, pd.DataFrame) and len(conformal_diagnostics):
        cd = conformal_diagnostics.copy()
        y_col = 'realised_flag_rate_pct' if 'realised_flag_rate_pct' in cd.columns else 'realized_flag_rate_pct'
        cd['target_alpha_pct'] = 100.0 * cd['alpha_target'].astype(float)

        sns.lineplot(data=cd, x='target_alpha_pct', y=y_col, marker='o', linewidth=1.8, label='Realised flag rate', ax=ax2)
        _x = np.sort(cd['target_alpha_pct'].to_numpy(dtype=float))
        sns.lineplot(data=pd.DataFrame({'target_alpha_pct': _x, 'ideal': _x}), x='target_alpha_pct', y='ideal',
                     linestyle='--', linewidth=1.3, color='black', label='Ideal (y=x)', ax=ax2)
        ax2.set_title('Conformal calibration')
        ax2.set_xlabel('Target alpha (%)')
        ax2.set_ylabel('Realised flag rate (%)')
        ax2.legend(loc='best')
    else:
        _blank(ax2, 'Conformal calibration', 'No conformal diagnostics available')

    # 3) Drift heatmap
    if isinstance(drift_results, pd.DataFrame) and len(drift_results):
        cols = [c for c in ['ks_stat', 'wasserstein', 'psi_10bins'] if c in drift_results.columns]
        if len(cols):
            raw = drift_results.set_index('score')[cols].astype(float)
            norm = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
            annot = raw.applymap(lambda v: f'{v:.3g}')
            sns.heatmap(norm, cmap='YlOrRd', vmin=0.0, vmax=1.0, annot=annot, fmt='', ax=ax3,
                        cbar_kws={'label': 'Relative intensity'})
            ax3.set_title('Drift intensity heatmap (relative)')
            ax3.set_xlabel('Metric')
            ax3.set_ylabel('Detector')
        else:
            _blank(ax3, 'Drift intensity heatmap (relative)', 'No drift columns available')
    else:
        _blank(ax3, 'Drift intensity heatmap (relative)', 'No drift diagnostics available')

    # 4) Model flag rates
    if isinstance(model_error_metrics, pd.DataFrame) and len(model_error_metrics):
        sns.barplot(data=model_error_metrics, x='detector', y='test_flag_rate_pct', ax=ax4)
        ax4.tick_params(axis='x', rotation=30)
        ax4.set_title('Model test flag rate')
        ax4.set_xlabel('Detector')
        ax4.set_ylabel('Flag rate (%)')
    else:
        _blank(ax4, 'Model test flag rate', 'No model error metrics available')

    # 5) Model agreement vs baseline
    if isinstance(model_error_metrics, pd.DataFrame) and len(model_error_metrics):
        mcols = [c for c in ['jaccard_vs_baseline', 'rank_corr_vs_baseline'] if c in model_error_metrics.columns]
        if len(mcols):
            mm = model_error_metrics.melt(id_vars='detector', value_vars=mcols, var_name='metric', value_name='value')
            sns.lineplot(data=mm, x='detector', y='value', hue='metric', marker='o', ax=ax5)
            ax5.tick_params(axis='x', rotation=30)
            ax5.set_ylim(0.0, 1.05)
            ax5.set_title('Model agreement vs baseline')
            ax5.set_xlabel('Detector')
            ax5.set_ylabel('Score')
        else:
            _blank(ax5, 'Model agreement vs baseline', 'No agreement columns found')
    else:
        _blank(ax5, 'Model agreement vs baseline', 'No model error metrics available')

    # 6) Feature lift (fallback: top-k overlap)
    if isinstance(feature_lift_results, pd.DataFrame) and len(feature_lift_results):
        top = feature_lift_results.head(12).sort_values('lift_vs_unflagged', ascending=False)
        sns.barplot(data=top, x='lift_vs_unflagged', y='feature', color='#4C78A8', ax=ax6)
        ax6.axvline(1.0, linestyle='--', color='black', linewidth=1)
        ax6.set_title('Top feature lifts (flagged vs unflagged)')
        ax6.set_xlabel('Lift ratio')
        ax6.set_ylabel('Feature')
    elif isinstance(detector_topk_overlap, pd.DataFrame) and len(detector_topk_overlap):
        dt = detector_topk_overlap.melt(id_vars='k', var_name='comparison', value_name='jaccard')
        sns.lineplot(data=dt, x='k', y='jaccard', hue='comparison', marker='o', ax=ax6)
        ax6.set_title('Detector top-k overlap')
        ax6.set_xlabel('k')
        ax6.set_ylabel('Jaccard overlap')
        ax6.set_ylim(0.0, 1.0)
    else:
        _blank(ax6, 'Feature lift / overlap', 'No feature-lift or overlap data available')

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=160, bbox_inches='tight')

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig


overview = no_label_dashboard.get('overview', pd.DataFrame())
if isinstance(overview, pd.DataFrame) and len(overview):
    print('Dashboard snapshot:')
    print(overview.to_string(index=False))

_ = render_no_label_dashboard(no_label_dashboard, save_path=None, show=True)

In [ ]:
# ── 8. Export monitoring artifacts (CSV + PNG) ───────────────────────────────
import os

if 'no_label_dashboard' not in globals():
    raise RuntimeError('Run section 6 (No-label evaluation dashboard) first.')
if 'render_no_label_dashboard' not in globals():
    raise RuntimeError('Run section 7 first (dashboard renderer missing).')

EXPORT_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
OUT_DIR = os.path.join(EXPORT_ROOT, 'monitoring')
os.makedirs(OUT_DIR, exist_ok=True)

export_rows = []

# 1) Export dashboard tables
for name, obj in no_label_dashboard.items():
    if isinstance(obj, pd.DataFrame):
        out_path = os.path.join(OUT_DIR, f'{name}.csv')
        obj.to_csv(out_path, index=(name == 'agreement_corr'))
        export_rows.append({'artifact': name, 'type': 'csv', 'path': out_path, 'rows': int(len(obj))})

# 2) Render one compact dashboard figure via shared renderer
fig_path = os.path.join(OUT_DIR, 'no_label_dashboard.png')
_ = render_no_label_dashboard(no_label_dashboard, save_path=fig_path, show=False)
export_rows.append({'artifact': 'no_label_dashboard', 'type': 'png', 'path': fig_path, 'rows': np.nan})

# 3) Save manifest
manifest = pd.DataFrame(export_rows)
manifest_path = os.path.join(OUT_DIR, 'monitoring_manifest.csv')
manifest.to_csv(manifest_path, index=False)

print('Export complete. Artifacts:')
print(manifest.to_string(index=False))
print(f'\nOutput folder: {OUT_DIR}')

In [ ]:
# ── 11. Main-report figures (high-signal, publication-ready) ─────────────────
import os


def _finite(a):
    x = np.asarray(a, dtype=float)
    return x[np.isfinite(x)]


def _psi_local(expected, actual, bins=10, eps=1e-6):
    exp = _finite(expected)
    act = _finite(actual)
    if len(exp) == 0 or len(act) == 0:
        return np.nan

    edges = np.quantile(exp, np.linspace(0, 1, bins + 1))
    edges = np.unique(edges)
    if len(edges) < 3:
        return 0.0

    exp_hist, _ = np.histogram(exp, bins=edges)
    act_hist, _ = np.histogram(act, bins=edges)

    exp_pct = np.clip(exp_hist / max(exp_hist.sum(), 1), eps, None)
    act_pct = np.clip(act_hist / max(act_hist.sum(), 1), eps, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def _conformal_rate_local(cal_scores, test_scores, alpha):
    cal = _finite(cal_scores)
    tst = _finite(test_scores)
    if len(cal) == 0 or len(tst) == 0:
        return np.nan, np.nan

    thr = float(np.quantile(cal, 1 - alpha))
    pvals = np.array([(np.sum(cal >= s) + 1) / (len(cal) + 1) for s in tst], dtype=float)
    rate = float((pvals <= alpha).mean())
    return thr, rate


def _plot_hist_kde(ax, data, color, label, alpha=0.35):
    x = _finite(data)
    if len(x) == 0:
        return

    sns.histplot(x=x, bins=45, stat='density', alpha=alpha, color=color, label=f'{label} hist', ax=ax)
    if len(np.unique(x)) > 3:
        try:
            sns.kdeplot(x=x, color=color, linewidth=2.0, label=f'{label} KDE', ax=ax)
        except Exception:
            pass


def _wallet_time_map(df):
    if {'wallet', 'block_timestamp', 'combined_tx_score'}.issubset(df.columns):
        idx = df.groupby('wallet')['combined_tx_score'].idxmax()
    elif {'wallet', 'block_timestamp', 'tr_tx_score'}.issubset(df.columns):
        idx = df.groupby('wallet')['tr_tx_score'].idxmax()
    elif {'wallet', 'block_timestamp'}.issubset(df.columns):
        idx = df.groupby('wallet')['block_timestamp'].idxmax()
    else:
        return pd.Series(dtype='datetime64[ns]')

    out = (
        df.loc[idx, ['wallet', 'block_timestamp']]
        .drop_duplicates('wallet')
        .set_index('wallet')['block_timestamp']
    )
    return pd.to_datetime(out, errors='coerce')


if 'wallet_scores' not in globals() or 'df_flat' not in globals():
    raise RuntimeError('Run earlier sections first (wallet_scores/df_flat missing).')

EXPORT_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
MAIN_FIG_DIR = os.path.join(EXPORT_ROOT, 'monitoring', 'main_report_figures')
os.makedirs(MAIN_FIG_DIR, exist_ok=True)
main_fig_manifest = []

alpha_main = float(ALPHA) if 'ALPHA' in globals() else 0.01

# Masks
if 'test_wallets' in globals():
    test_mask_main = wallet_scores['wallet'].isin(test_wallets).to_numpy()
else:
    test_mask_main = np.ones(len(wallet_scores), dtype=bool)

if 'val_wallets' in globals():
    val_mask_main = wallet_scores['wallet'].isin(val_wallets).to_numpy()
else:
    val_mask_main = ~test_mask_main

if val_mask_main.sum() == 0:
    val_mask_main = test_mask_main.copy()

# Common vectors
for col in ['if_score', 'tr_score', 'raw_score']:
    if col not in wallet_scores.columns:
        raise RuntimeError(f'Missing required column in wallet_scores: {col}')

if 'conformal_flag' in wallet_scores.columns:
    flag_main = wallet_scores['conformal_flag'].fillna(0).astype(int).to_numpy()
else:
    flag_main = np.zeros(len(wallet_scores), dtype=int)

if 'n_tx' in wallet_scores.columns:
    n_tx_main = wallet_scores['n_tx'].fillna(1).to_numpy(dtype=float)
else:
    n_tx_main = np.ones(len(wallet_scores), dtype=float)

# ── Figure 1: Score separation (hist + KDE) ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=False)
score_specs = [
    ('if_score', 'Isolation Forest score'),
    ('tr_score', 'Transformer AE score'),
    ('raw_score', 'Combined score'),
]

for ax, (col, title) in zip(axes, score_specs):
    val_x = _finite(wallet_scores.loc[val_mask_main, col].to_numpy(dtype=float))
    tst_x = _finite(wallet_scores.loc[test_mask_main, col].to_numpy(dtype=float))

    _plot_hist_kde(ax, val_x, color='#4C78A8', label='Val')
    _plot_hist_kde(ax, tst_x, color='#F58518', label='Test')

    if len(val_x):
        thr = float(np.quantile(val_x, 1 - alpha_main))
        ax.axvline(thr, linestyle='--', linewidth=1.6, color='black', label=f'Val q(1-a), a={alpha_main:.3f}')

    ax.set_title(title)
    ax.set_xlabel(col)
    ax.grid(alpha=0.20)

axes[0].set_ylabel('Density')
handles, labels = axes[-1].get_legend_handles_labels()
if len(handles):
    axes[-1].legend(loc='best', fontsize=8)

fig.suptitle('Score separation: validation vs test', y=1.02)
plt.tight_layout()
fig1_path = os.path.join(MAIN_FIG_DIR, '01_score_separation_hist_kde.png')
fig.savefig(fig1_path, dpi=170, bbox_inches='tight')
plt.show()
main_fig_manifest.append({'figure': 'score_separation_hist_kde', 'path': fig1_path})

# ── Figure 2: Detector agreement map ─────────────────────────────────────────
plot_df = wallet_scores.loc[test_mask_main, ['wallet', 'if_score', 'tr_score', 'raw_score']].copy()
plot_df['flag'] = flag_main[test_mask_main]
plot_df['n_tx'] = n_tx_main[test_mask_main]

if len(plot_df):
    plot_df = plot_df.copy()
    plot_df['flag_label'] = np.where(plot_df['flag'].astype(int) == 1, 'Flagged', 'Unflagged')
    plot_df['size_scaled'] = 20 + 90 * (
        np.log1p(plot_df['n_tx'].to_numpy()) / np.log1p(max(plot_df['n_tx'].max(), 1))
    )

    fig, ax = plt.subplots(figsize=(7.6, 6.2))
    sns.scatterplot(
        data=plot_df,
        x='if_score',
        y='tr_score',
        hue='flag_label',
        size='size_scaled',
        sizes=(20, 110),
        alpha=0.70,
        ax=ax,
    )

    thr_if = float(np.quantile(_finite(wallet_scores.loc[val_mask_main, 'if_score']), 1 - alpha_main))
    thr_tr = float(np.quantile(_finite(wallet_scores.loc[val_mask_main, 'tr_score']), 1 - alpha_main))
    ax.axvline(thr_if, linestyle='--', linewidth=1.2, color='gray')
    ax.axhline(thr_tr, linestyle='--', linewidth=1.2, color='gray')

    ax.set_title('Detector agreement map (test wallets)')
    ax.set_xlabel('if_score')
    ax.set_ylabel('tr_score')
    ax.legend(loc='best')

    fig2_path = os.path.join(MAIN_FIG_DIR, '02_detector_agreement_scatter.png')
    fig.savefig(fig2_path, dpi=170, bbox_inches='tight')
    plt.show()
    main_fig_manifest.append({'figure': 'detector_agreement_scatter', 'path': fig2_path})

# ── Figure 3: Conformal calibration curve ────────────────────────────────────
if 'no_label_dashboard' in globals() and isinstance(no_label_dashboard.get('conformal_diagnostics', None), pd.DataFrame):
    conf_df = no_label_dashboard['conformal_diagnostics'].copy()
else:
    alpha_grid = [0.005, 0.01, 0.02, 0.05]
    conf_rows = []
    cal = wallet_scores.loc[val_mask_main, 'raw_score'].to_numpy(dtype=float)
    tst = wallet_scores.loc[test_mask_main, 'raw_score'].to_numpy(dtype=float)
    for a in alpha_grid:
        thr, rr = _conformal_rate_local(cal, tst, a)
        conf_rows.append({'alpha_target': float(a), 'threshold': thr, 'realized_flag_rate_pct': float(rr * 100)})
    conf_df = pd.DataFrame(conf_rows)

if len(conf_df):
    conf_df = conf_df.copy()
    y_col = 'realised_flag_rate_pct' if 'realised_flag_rate_pct' in conf_df.columns else 'realized_flag_rate_pct'
    conf_df['target_alpha_pct'] = 100.0 * conf_df['alpha_target'].astype(float)

    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    sns.lineplot(data=conf_df, x='target_alpha_pct', y=y_col, marker='o', linewidth=2.0, label='Realised', ax=ax)
    _x = np.sort(conf_df['target_alpha_pct'].to_numpy(dtype=float))
    _ideal = pd.DataFrame({'target_alpha_pct': _x, 'ideal': _x})
    sns.lineplot(data=_ideal, x='target_alpha_pct', y='ideal', linestyle='--', linewidth=1.3, color='black', label='Ideal y=x', ax=ax)
    ax.set_title('Conformal calibration (wallet-level)')
    ax.set_xlabel('Target alpha (%)')
    ax.set_ylabel('Realised flagged rate (%)')
    ax.legend(loc='best')

    fig3_path = os.path.join(MAIN_FIG_DIR, '03_conformal_calibration_curve.png')
    fig.savefig(fig3_path, dpi=170, bbox_inches='tight')
    plt.show()
    main_fig_manifest.append({'figure': 'conformal_calibration_curve', 'path': fig3_path})

# Shared wallet->time map
wallet_ts_map = _wallet_time_map(df_flat)

# ── Figure 4: Temporal monitoring trend ──────────────────────────────────────
trend_df = wallet_scores.loc[test_mask_main, ['wallet', 'raw_score']].copy()
trend_df['flag'] = flag_main[test_mask_main]
trend_df['top_tx_ts'] = pd.to_datetime(trend_df['wallet'].map(wallet_ts_map), errors='coerce')
trend_df = trend_df.dropna(subset=['top_tx_ts'])

if len(trend_df):
    trend_df['week'] = trend_df['top_tx_ts'].dt.to_period('W').dt.start_time

    weekly_flagged = trend_df.groupby('week')['flag'].sum().sort_index()
    weekly_med_raw = trend_df.groupby('week')['raw_score'].median().sort_index()
    weekly_med_roll = weekly_med_raw.rolling(4, min_periods=1).median()

    _trend_plot = pd.DataFrame({
        'week': weekly_flagged.index,
        'flagged_wallets': weekly_flagged.values,
        'rolling_median_raw_score': weekly_med_roll.reindex(weekly_flagged.index).values,
    })

    fig, ax1 = plt.subplots(figsize=(10.2, 5.0))
    sns.lineplot(data=_trend_plot, x='week', y='flagged_wallets', marker='o', linewidth=1.8, label='Flagged wallets / week', ax=ax1)
    ax1.set_xlabel('Week')
    ax1.set_ylabel('Flagged wallets')

    ax2 = ax1.twinx()
    sns.lineplot(
        data=_trend_plot,
        x='week',
        y='rolling_median_raw_score',
        marker='s',
        linewidth=2.0,
        color='#E45756',
        label='Rolling median raw_score (4w)',
        ax=ax2,
    )
    ax2.set_ylabel('Raw score (median)')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    ax1.set_title('Temporal monitoring: weekly flagged volume and score trend')

    fig4_path = os.path.join(MAIN_FIG_DIR, '04_temporal_monitoring_trend.png')
    fig.savefig(fig4_path, dpi=170, bbox_inches='tight')
    plt.show()
    main_fig_manifest.append({'figure': 'temporal_monitoring_trend', 'path': fig4_path})

# ── Figure 5: Anomaly-type composition over time ─────────────────────────────
if 'semantic_anomaly_type' in wallet_scores.columns:
    comp_df = wallet_scores.loc[test_mask_main, ['wallet', 'semantic_anomaly_type']].copy()
    comp_df['flag'] = flag_main[test_mask_main]
    comp_df['top_tx_ts'] = pd.to_datetime(comp_df['wallet'].map(wallet_ts_map), errors='coerce')
    comp_df = comp_df.dropna(subset=['top_tx_ts'])
    comp_df = comp_df[(comp_df['flag'] == 1) & comp_df['semantic_anomaly_type'].notna()]

    if len(comp_df):
        comp_df['week'] = comp_df['top_tx_ts'].dt.to_period('W').dt.start_time
        type_week = comp_df.groupby(['week', 'semantic_anomaly_type']).size().unstack(fill_value=0).sort_index()
        type_week = type_week.loc[type_week.sum(axis=1) > 0]

        if len(type_week):
            comp_pct = (type_week.div(type_week.sum(axis=1), axis=0) * 100).reset_index()
            comp_long = comp_pct.melt(id_vars='week', var_name='semantic_anomaly_type', value_name='composition_pct')

            fig, ax = plt.subplots(figsize=(10.2, 5.2))
            sns.lineplot(data=comp_long, x='week', y='composition_pct', hue='semantic_anomaly_type', marker='o', ax=ax)
            ax.set_title('Flagged anomaly-type composition over time')
            ax.set_xlabel('Week')
            ax.set_ylabel('Composition (%)')
            ax.set_ylim(0, 100)
            ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))

            fig5_path = os.path.join(MAIN_FIG_DIR, '05_anomaly_type_composition_weekly.png')
            fig.savefig(fig5_path, dpi=170, bbox_inches='tight')
            plt.show()
            main_fig_manifest.append({'figure': 'anomaly_type_composition_weekly', 'path': fig5_path})

# ── Figure 6: Top feature-contributor heatmap ────────────────────────────────
if all(v in globals() for v in ['per_feat_errors', 'tx_feature_cols']) and isinstance(per_feat_errors, np.ndarray):
    if 'combined_tx_score' in df_flat.columns:
        top_tx_idx_map = df_flat.groupby('wallet')['combined_tx_score'].idxmax()
    elif 'tr_tx_score' in df_flat.columns:
        top_tx_idx_map = df_flat.groupby('wallet')['tr_tx_score'].idxmax()
    else:
        top_tx_idx_map = df_flat.reset_index().groupby('wallet')['index'].first()

    flagged_wallets = wallet_scores.loc[test_mask_main].copy()
    flagged_wallets['flag'] = flag_main[test_mask_main]
    flagged_wallets = flagged_wallets[flagged_wallets['flag'] == 1]

    if len(flagged_wallets):
        top_wallets = flagged_wallets.sort_values('raw_score', ascending=False).head(25)['wallet']
        tx_lookup = top_tx_idx_map.reindex(top_wallets).dropna()
        tx_idx = tx_lookup.astype(int).to_numpy()

        if len(tx_idx):
            mat = per_feat_errors[tx_idx]
            n_show_feat = int(min(12, mat.shape[1]))
            feat_ids = np.argsort(mat.mean(axis=0))[-n_show_feat:]
            feat_names = [tx_feature_cols[i] for i in feat_ids]
            mat_show = np.log1p(mat[:, feat_ids])

            row_std = mat_show.std(axis=1, keepdims=True) + 1e-9
            mat_norm = (mat_show - mat_show.mean(axis=1, keepdims=True)) / row_std

            wallet_labels = []
            for w in tx_lookup.index.astype(str):
                wallet_labels.append(w if len(w) <= 14 else f"{w[:6]}...{w[-4:]}")

            fig, ax = plt.subplots(figsize=(11.8, 7.2))
            sns.heatmap(
                mat_norm,
                cmap='coolwarm',
                xticklabels=feat_names,
                yticklabels=wallet_labels,
                cbar_kws={'label': 'Row-normalized log reconstruction error'},
                ax=ax,
            )
            ax.set_title('Top feature contributors (flagged wallets)')
            ax.set_xlabel('Feature')
            ax.set_ylabel('Flagged wallets (top raw_score)')
            ax.tick_params(axis='x', rotation=45)

            fig6_path = os.path.join(MAIN_FIG_DIR, '06_feature_contributor_heatmap.png')
            fig.savefig(fig6_path, dpi=170, bbox_inches='tight')
            plt.show()
            main_fig_manifest.append({'figure': 'feature_contributor_heatmap', 'path': fig6_path})

main_report_figures_manifest = pd.DataFrame(main_fig_manifest)
print('Main-report figures generated:')
print(main_report_figures_manifest.to_string(index=False) if len(main_report_figures_manifest) else 'No figures generated.')
print(f'\nOutput folder: {MAIN_FIG_DIR}')

if 'no_label_dashboard' in globals() and isinstance(main_report_figures_manifest, pd.DataFrame):
    no_label_dashboard['main_report_figures_manifest'] = main_report_figures_manifest

In [ ]:
# ── 12. Appendix visual pack (robustness + ops diagnostics) ─────────────────
import os
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


EXPORT_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
APP_FIG_DIR = os.path.join(EXPORT_ROOT, 'monitoring', 'appendix_figures')
os.makedirs(APP_FIG_DIR, exist_ok=True)
appendix_manifest = []


def _finite_local(a):
    x = np.asarray(a, dtype=float)
    return x[np.isfinite(x)]


def _psi_month(expected, actual, bins=10, eps=1e-6):
    exp = _finite_local(expected)
    act = _finite_local(actual)
    if len(exp) == 0 or len(act) == 0:
        return np.nan

    edges = np.quantile(exp, np.linspace(0, 1, bins + 1))
    edges = np.unique(edges)
    if len(edges) < 3:
        return 0.0

    exp_hist, _ = np.histogram(exp, bins=edges)
    act_hist, _ = np.histogram(act, bins=edges)

    exp_pct = np.clip(exp_hist / max(exp_hist.sum(), 1), eps, None)
    act_pct = np.clip(act_hist / max(act_hist.sum(), 1), eps, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def _wallet_time_map_appendix(df):
    if {'wallet', 'block_timestamp', 'combined_tx_score'}.issubset(df.columns):
        idx = df.groupby('wallet')['combined_tx_score'].idxmax()
    elif {'wallet', 'block_timestamp', 'tr_tx_score'}.issubset(df.columns):
        idx = df.groupby('wallet')['tr_tx_score'].idxmax()
    elif {'wallet', 'block_timestamp'}.issubset(df.columns):
        idx = df.groupby('wallet')['block_timestamp'].idxmax()
    else:
        return pd.Series(dtype='datetime64[ns]')

    out = (
        df.loc[idx, ['wallet', 'block_timestamp']]
        .drop_duplicates('wallet')
        .set_index('wallet')['block_timestamp']
    )
    return pd.to_datetime(out, errors='coerce')


# A1) Hyperparameter heatmaps (from section 3 sensitivity sweep)
if 'sensitivity_results' in globals() and isinstance(sensitivity_results, pd.DataFrame) and len(sensitivity_results):
    piv_flag = sensitivity_results.pivot(index='alpha', columns='w_if', values='flag_rate_pct').sort_index()
    piv_jac = sensitivity_results.pivot(index='alpha', columns='w_if', values='jaccard_vs_baseline').sort_index()

    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))
    sns.heatmap(
        piv_flag,
        cmap='viridis',
        annot=True,
        fmt='.2f',
        cbar_kws={'label': 'Flag rate (%)'},
        ax=axes[0],
    )
    axes[0].set_title('Flag rate (%)')
    axes[0].set_xlabel('w_if')
    axes[0].set_ylabel('alpha')

    sns.heatmap(
        piv_jac,
        cmap='magma',
        annot=True,
        fmt='.3f',
        cbar_kws={'label': 'Jaccard'},
        ax=axes[1],
    )
    axes[1].set_title('Jaccard vs baseline')
    axes[1].set_xlabel('w_if')
    axes[1].set_ylabel('alpha')

    plt.tight_layout()
    a1_path = os.path.join(APP_FIG_DIR, 'A1_hyperparameter_heatmaps.png')
    fig.savefig(a1_path, dpi=170, bbox_inches='tight')
    plt.show()
    appendix_manifest.append({'figure': 'A1_hyperparameter_heatmaps', 'path': a1_path})

# A2) IF contamination sweep curves (from section 9)
if 'if_sensitivity_results' in globals() and isinstance(if_sensitivity_results, pd.DataFrame) and len(if_sensitivity_results):
    d = if_sensitivity_results.sort_values('contamination').copy()
    d['jaccard_vs_ref_pct'] = 100.0 * d['jaccard_vs_ref'] if 'jaccard_vs_ref' in d.columns else np.nan
    d['rank_corr_vs_ref_pct'] = 100.0 * d['rank_corr_vs_ref'] if 'rank_corr_vs_ref' in d.columns else np.nan

    pcols = [c for c in ['flag_rate_pct', 'jaccard_vs_ref_pct', 'rank_corr_vs_ref_pct'] if c in d.columns]
    dm = d.melt(id_vars='contamination', value_vars=pcols, var_name='metric', value_name='percent')

    fig, ax = plt.subplots(figsize=(8.6, 5.0))
    sns.lineplot(data=dm, x='contamination', y='percent', hue='metric', marker='o', ax=ax)
    ax.set_xscale('log')
    ax.set_title('A2. IF contamination sensitivity')
    ax.set_xlabel('contamination')
    ax.set_ylabel('Percent')
    ax.legend(loc='best')

    plt.tight_layout()
    a2_path = os.path.join(APP_FIG_DIR, 'A2_if_contamination_sensitivity.png')
    fig.savefig(a2_path, dpi=170, bbox_inches='tight')
    plt.show()
    appendix_manifest.append({'figure': 'A2_if_contamination_sensitivity', 'path': a2_path})

# A3) Embedding projection (UMAP if available, else t-SNE, else PCA)
projection_meta = pd.DataFrame()
if 'X_emb' in globals() and 'wallet_rep' in globals():
    X_all = np.asarray(X_emb, dtype=float)
    w_all = np.asarray(wallet_rep)

    if len(X_all) >= 20 and len(w_all) == len(X_all):
        rng = np.random.default_rng(42)
        n_plot = int(min(3000, len(X_all)))
        idx = rng.choice(len(X_all), size=n_plot, replace=False) if len(X_all) > n_plot else np.arange(len(X_all))

        X_plot = X_all[idx]
        w_plot = w_all[idx]

        # Cluster coloring
        if 'wallet_embedding_clusters' in globals() and isinstance(wallet_embedding_clusters, pd.DataFrame) and len(wallet_embedding_clusters):
            cl_map = wallet_embedding_clusters.set_index('wallet')['cluster']
            cluster = pd.Series(w_plot).map(cl_map).fillna(-1).astype(int).to_numpy()
        else:
            k_guess = 4
            if 'transformer_embedding_meta' in globals() and isinstance(transformer_embedding_meta, pd.DataFrame) and len(transformer_embedding_meta):
                if 'best_k' in transformer_embedding_meta.columns:
                    k_guess = int(transformer_embedding_meta['best_k'].iloc[0])
            km_tmp = KMeans(n_clusters=max(2, min(k_guess, len(X_plot) - 1)), random_state=42, n_init=10)
            cluster = km_tmp.fit_predict(X_plot)

        # Flag overlay
        if 'conformal_flag' in wallet_scores.columns:
            f_map = wallet_scores.set_index('wallet')['conformal_flag'].fillna(0).astype(int)
            flag = pd.Series(w_plot).map(f_map).fillna(0).astype(int).to_numpy()
        else:
            flag = np.zeros(len(w_plot), dtype=int)

        method = 'PCA'
        reducer_note = ''
        try:
            import umap
            reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=25, min_dist=0.10)
            XY = reducer.fit_transform(X_plot)
            method = 'UMAP'
        except Exception as e_umap:
            try:
                XY = TSNE(n_components=2, random_state=42, perplexity=30, init='pca').fit_transform(X_plot)
                method = 't-SNE'
            except Exception as e_tsne:
                XY = PCA(n_components=2, random_state=42).fit_transform(X_plot)
                method = 'PCA'
                reducer_note = f'Fallback due to UMAP/t-SNE errors: {str(e_umap)[:80]} | {str(e_tsne)[:80]}'

        _proj_df = pd.DataFrame({
            'x': XY[:, 0],
            'y': XY[:, 1],
            'cluster': cluster.astype(str),
            'flag': flag.astype(int),
        })

        fig, ax = plt.subplots(figsize=(7.2, 6.0))
        sns.scatterplot(data=_proj_df, x='x', y='y', hue='cluster', palette='tab20', s=18, alpha=0.75, linewidth=0, ax=ax)
        if np.sum(flag == 1):
            sns.scatterplot(
                data=_proj_df[_proj_df['flag'] == 1],
                x='x',
                y='y',
                s=52,
                facecolor='none',
                edgecolor='red',
                linewidth=0.9,
                legend=False,
                ax=ax,
            )

        ax.set_title(f'A3. Wallet embedding projection ({method})')
        ax.set_xlabel('Component 1')
        ax.set_ylabel('Component 2')

        plt.tight_layout()
        a3_path = os.path.join(APP_FIG_DIR, 'A3_embedding_projection.png')
        fig.savefig(a3_path, dpi=170, bbox_inches='tight')
        plt.show()
        appendix_manifest.append({'figure': 'A3_embedding_projection', 'path': a3_path})

        projection_meta = pd.DataFrame([{
            'method': method,
            'n_points_plotted': int(len(X_plot)),
            'flagged_points': int(np.sum(flag == 1)),
            'note': reducer_note,
        }])

# A4) Rolling drift panel by month (raw_score)
monthly_drift = pd.DataFrame()
if 'wallet_scores' in globals() and 'df_flat' in globals() and 'raw_score' in wallet_scores.columns:
    tmap = _wallet_time_map_appendix(df_flat)
    wd = wallet_scores[['wallet', 'raw_score']].copy()

    if 'test_wallets' in globals():
        wd = wd[wd['wallet'].isin(test_wallets)]

    wd['ts'] = pd.to_datetime(wd['wallet'].map(tmap), errors='coerce')
    wd = wd.dropna(subset=['ts'])

    if len(wd):
        wd['month'] = wd['ts'].dt.to_period('M').dt.to_timestamp()
        months = sorted(wd['month'].dropna().unique())

        rows = []
        prev = None
        for m in months:
            cur = _finite_local(wd.loc[wd['month'] == m, 'raw_score'].to_numpy(dtype=float))
            if prev is None:
                prev = cur
                continue
            if len(prev) == 0 or len(cur) == 0:
                prev = cur
                continue

            ks_stat, ks_p = ks_2samp(prev, cur)
            rows.append({
                'month': pd.Timestamp(m),
                'ks_stat': float(ks_stat),
                'ks_pvalue': float(ks_p),
                'wasserstein': float(wasserstein_distance(prev, cur)),
                'psi': float(_psi_month(prev, cur, bins=10)),
            })
            prev = cur

        monthly_drift = pd.DataFrame(rows)

        if len(monthly_drift):
            mdm = monthly_drift.melt(id_vars='month', value_vars=['ks_stat', 'wasserstein', 'psi'], var_name='metric', value_name='value')
            fig, ax = plt.subplots(figsize=(10.2, 4.8))
            sns.lineplot(data=mdm, x='month', y='value', hue='metric', marker='o', ax=ax)
            ax.set_title('A4. Rolling monthly drift (raw_score, month-to-month)')
            ax.set_xlabel('Month')
            ax.set_ylabel('Drift metric value')
            ax.legend(loc='best')

            plt.tight_layout()
            a4_path = os.path.join(APP_FIG_DIR, 'A4_monthly_drift_panel.png')
            fig.savefig(a4_path, dpi=170, bbox_inches='tight')
            plt.show()
            appendix_manifest.append({'figure': 'A4_monthly_drift_panel', 'path': a4_path})

# A5) Alert budget curve
if 'raw_score' in wallet_scores.columns:
    if 'test_wallets' in globals():
        score_budget = _finite_local(wallet_scores.loc[wallet_scores['wallet'].isin(test_wallets), 'raw_score'].to_numpy(dtype=float))
    else:
        score_budget = _finite_local(wallet_scores['raw_score'].to_numpy(dtype=float))

    if len(score_budget):
        s = np.sort(score_budget)[::-1]
        k = np.arange(1, len(s) + 1)
        cum_mass_pct = 100.0 * np.cumsum(s) / (np.sum(s) + 1e-12)
        wallet_pct = 100.0 * k / len(s)

        curve_df = pd.DataFrame({'wallet_pct': wallet_pct, 'cum_mass_pct': cum_mass_pct})

        fig, ax = plt.subplots(figsize=(8.2, 5.2))
        sns.lineplot(data=curve_df, x='wallet_pct', y='cum_mass_pct', linewidth=2.2, label='Cumulative anomaly mass', ax=ax)
        sns.lineplot(data=curve_df, x='wallet_pct', y='wallet_pct', linestyle='--', linewidth=1.2, label='Uniform baseline', ax=ax)

        for p in [1, 5, 10]:
            idx_p = max(1, int(np.ceil((p / 100.0) * len(s)))) - 1
            _x = wallet_pct[idx_p]
            _y = cum_mass_pct[idx_p]
            sns.scatterplot(x=[_x], y=[_y], s=40, color='black', ax=ax, legend=False)
            ax.text(_x, _y, f' Top {p}%: {_y:.1f}%', fontsize=8, va='bottom')

        ax.set_title('A5. Alert budget curve (top-K wallet concentration)')
        ax.set_xlabel('Top wallets reviewed (%)')
        ax.set_ylabel('Captured anomaly-score mass (%)')
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.legend(loc='lower right')

        plt.tight_layout()
        a5_path = os.path.join(APP_FIG_DIR, 'A5_alert_budget_curve.png')
        fig.savefig(a5_path, dpi=170, bbox_inches='tight')
        plt.show()
        appendix_manifest.append({'figure': 'A5_alert_budget_curve', 'path': a5_path})

appendix_figures_manifest = pd.DataFrame(appendix_manifest)
print('Appendix figures generated:')
print(appendix_figures_manifest.to_string(index=False) if len(appendix_figures_manifest) else 'No figures generated.')
print(f'\nOutput folder: {APP_FIG_DIR}')

if 'no_label_dashboard' in globals():
    no_label_dashboard['appendix_figures_manifest'] = appendix_figures_manifest
    if len(monthly_drift):
        no_label_dashboard['monthly_drift'] = monthly_drift
    if len(projection_meta):
        no_label_dashboard['embedding_projection_meta'] = projection_meta

## Operational Context & Ethical Considerations

### Deployment assumptions
| Parameter | Value / Assumption |
|-----------|-------------------|
| Target FPR | ≤ α = 1 % (conformal guarantee, subject to exchangeability) |
| Minimum wallets needed to flag | 1 anomalous transaction (max aggregation) |
| Downstream action | Risk-tier assignment (low / medium / high / critical) |
| Human review required | Yes — no automated blocking |

### False-positive risk
- Flagging a wallet as anomalous has real-world consequences (account freezing,
  compliance investigations, reputational harm).
- At α = 0.01 the expected FPR is ≤ 1 % **only** if the exchangeability assumption
  holds.  The temporal KS test in the conformal validity section should be checked
  before each deployment.
- Any downstream action beyond risk-scoring **must** include a human review step.

### Label-free limitations
Because the system is fully unsupervised, there is no precision or recall estimate
against a ground-truth anomaly set.  The synthetic contamination test (Cell 15) is
a necessary proxy but not a substitute for a labelled evaluation:
- Detection rate on synthetic anomalies does **not** guarantee detection of novel
  real-world attack patterns.
- Consider enriching the evaluation with publicly available on-chain scam/exploit
  address lists (e.g., Solana Foundation reports, public MEV bot registries) as a
  held-out evaluation set without retraining.

### Scope
This model covers Solana mainnet transactions from September – November 2023.
It should **not** be applied to:
- Earlier periods without re-calibrating the conformal threshold.
- Other chains without retraining on chain-specific features.
- Real-time streaming without drift monitoring (see Cell 17 dashboard).

### References
- Tibshirani et al. (2019). *Conformal Prediction Under Covariate Shift.*
  NeurIPS 2019.
- Liu et al. (2008). *Isolation Forest.*  ICDM 2008.
- Chalapathy & Chawla (2019). *Deep Learning for Anomaly Detection: A Survey.*
  arXiv:1901.03407.
